# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


## Approach

Before any code, this section works out what the business problem asks of a model, because that reasoning
decides how the model is built, how it is judged, and what the app does. Each step names the section that
carries it out; no result is claimed here that the notebook does not compute later.

### 1. The decision that has to be made

The sales team cannot call everyone about the Wellness Tourism Package. Time and budget allow a limited number
of calls, and today the selection is made by hand, which the problem statement describes as inconsistent and
error-prone.

So the question is not really "will this customer buy?". It is "given that we can make N calls, which customers
should they be?". That is a question about priority, not about a yes or no per customer, and it changes what
the model has to deliver.

### 2. Therefore: a score per customer, not a yes/no

To rank customers, every customer needs a number that can be compared with every other customer. So the model
must return a score, and the customers with the highest scores are called first.

A yes/no answer would not do: if it says yes for 2,000 customers and the team can call 500, the model has not
answered the question. A score keeps the ordering, and where the line is drawn is decided separately (step 5).

The score expresses how much a customer resembles customers who bought before. It is not a probability of
buying: nothing in the training procedure forces it onto that scale. What can be said about it is measured per
group of customers, in step 6.

### 3. Therefore: only information from before the first contact

The call list is made *before* anyone is contacted. Anything the model needs must be available at that moment,
which rules out the four columns in the data description that describe the sales pitch: how long it lasted, how
many follow-ups there were, how satisfied the customer was, which product was pitched.

Those columns are strongly related to the purchase, so a model using them would look excellent. It would also
be useless, since the values do not exist yet when the list is made. *Data Preparation* removes them, along
with the customer id, which identifies a customer but says nothing about them.

### 4. Therefore: a metric that judges the whole ranking

A ranking cannot be judged by counting correct answers. Most customers do not buy, so a model that predicts
"no buyer" for everyone is right most of the time while producing an empty call list: accuracy rewards exactly
the behaviour that is useless here. How lopsided the data is follows from the target column and is reported in
*Data Preparation*.

A metric is also needed that does not depend on a cut-off, because the cut-off is a budget decision that has
not been made yet. That leads to PR-AUC (average precision), which summarises how well buyers are pushed to the
top across the whole ranking. *Model Training* uses it as the deciding metric, together with two safeguards: a
model that knows nothing (always the same score) as the floor every candidate must beat, and ROC-AUC as an
upper bound — profile data alone cannot plausibly predict that well, so an unexpectedly high value points to
information that leaked in rather than to a good model.

Only one metric decides. With several, any candidate can be made to win afterwards by choosing the metric that
happens to suit it.

### 5. Therefore: the cut-off is a separate, business-driven choice

Ranking the customers does not yet say whom to call. That needs a threshold: call everyone at or above it.
Where it lies depends on how many calls marketing can make, so *Model Training* fixes several thresholds — one
for calling the top 5%, 10% or 20% of customers, and one for the best balance between the two errors below —
computed on training data only.

For each threshold, two things matter and they pull against each other:

- *precision*: of the customers called, which share buys — how much calling effort is wasted;
- *recall*: of all buyers, which share is called — how much revenue is left on the table.

Calling more customers raises recall and lowers precision. F1 combines the two into one number, and *lift*
divides precision by the share of buyers in the data as a whole, which answers the question marketing actually
asks: how many times better is this than calling at random? Lift 1 means the model adds nothing.

### 6. Therefore: measure once, on customers the model never saw

The numbers above must say something about new customers, not about the customers used to build the model.
*Data Preparation* therefore sets aside a test set, and *Model Training* uses it once, after the model and the
thresholds are fixed, and reports the results with ranges that show how much they could shift by chance.

Selecting and measuring stay separate throughout: candidates are compared with cross-validation inside the
training data, and the thresholds are derived there as well. If the test set influenced those choices, its
result would describe the choices instead of predicting what happens with new customers.

### 7. Therefore: an app that scores one lead

The last step makes all of this usable: *Deployment* builds a Streamlit app in which a sales employee enters
the profile of one lead — a customer not yet contacted. The app computes the score with exactly the registered
model, compares it with the thresholds from step 5, and shows which contact group the lead falls into and what
that group achieved on the test set, so the answer comes with the evidence behind it.

# Model Building

In [30]:
# === PHASE 01a: PROJECT FOLDER ===
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

print(f"Project folder ready: {os.path.abspath('tourism_project')}")

Project folder ready: /mnt/c/Users/richa/Mijn Drive/tourism/tourism_project


In [31]:
# === PHASE 01b: MODEL BUILDING FOLDER ===
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

print("Model building folder ready: tourism_project/model_building")

Model building folder ready: tourism_project/model_building


### How this notebook and the pipeline fit together

The goal is to predict, before the sales team contacts a customer, whether that customer is
likely to buy the Wellness Tourism Package (`ProdTaken`). Marketing can then spend its calls on
the most promising customers. The pipeline consists of stages; each stage uses what the previous
stage produced:

| Stage | Script | Reads | Produces |
|---|---|---|---|
| Data registration | `model_building/data_register.py` | `tourism.csv` | raw data in a private Hugging Face dataset repository |
| Data preparation | `model_building/prep.py` | raw data from the Hub | `train.csv` and `test.csv` in the same repository |
| Model training | `model_building/train.py` | train and test from the Hub | the best model in a Hugging Face model repository, experiments in MLflow |
| Deployment and hosting | `deployment/`, `hosting/` | the model from the Hub | a Streamlit app in a public Hugging Face Space |

Every stage is a Python script. This notebook writes each script with `%%writefile` and runs it
with `%run -m`. Once the files are pushed to GitHub, GitHub Actions runs the same scripts with
`python -m` whenever the code changes. That repetition without manual steps is what turns the
scripts into an automated MLOps pipeline. The Hugging Face Hub is the shared storage between the
stages: each stage reads its input from the Hub and writes its output back, so a stage behaves
the same in Colab and in GitHub Actions.

### Setup

The next cell prepares the runtime and works in both places the notebook runs. In Colab it installs
MLflow, which Colab does not include, and reads the Hugging Face token from Colab *Secrets* (key icon
in the left sidebar → add `HF_TOKEN` → allow notebook access). Run locally, it skips both: the packages
are already installed and the token comes from `hf auth login` or the environment variable `HF_TOKEN`.
A token is never typed into a cell, because this notebook is published in a public GitHub repository;
the cell prints whether a token was found, never the token itself.

That token needs write access, because the scripts create and update the private repositories.

The cell after it writes `config.py`: one `Config` dataclass with the repository names, file names,
target and feature columns, and the constants of each stage. Every script imports it, so each
setting is defined in exactly one place.

In [32]:
# === PHASE 02: COLAB SETUP ===
# Colab only: install missing libraries and read the Hugging Face token from Colab Secrets.
# Locally the libraries are already installed and the token comes from `hf auth login`.
import sys

in_colab = "google.colab" in sys.modules
if in_colab:
    # -U: Colab ships an older huggingface_hub; the scripts need a recent one (e.g. for private repos).
    # If an import fails after this install, use Runtime -> Restart session and run this cell again.
    !pip install -q -U mlflow huggingface_hub
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Report the environment. The token itself is never printed, only whether one was found.
from importlib.metadata import version

import huggingface_hub

token_found = huggingface_hub.get_token() is not None
print(f"Running in Colab    : {in_colab}")
print(f"Hugging Face token  : {'found' if token_found else 'NOT found - add HF_TOKEN to Colab Secrets'}")
for package in ("pandas", "scikit-learn", "xgboost", "huggingface_hub", "mlflow"):
    print(f"{package:<20}: {version(package)}")

Running in Colab    : False
Hugging Face token  : found
pandas              : 3.0.3
scikit-learn        : 1.9.1
xgboost             : 3.4.1
huggingface_hub     : 1.31.0
mlflow              : 3.16.0


In [33]:
%%writefile tourism_project/config.py
"""Central configuration of the Visit with Us MLOps pipeline.

Context
    Every stage of the pipeline (data registration, data preparation, and later model training,
    deployment and hosting) imports this one Config dataclass. Paths, Hugging Face repository
    names, the target and feature columns, and the constants of each stage are defined here
    once. A change is therefore made in one place and used by every stage, in the notebook as
    well as in GitHub Actions.

    The notebook writes this file with %%writefile; GitHub Actions uses the copy that is
    committed to the repository. Settings are grouped per stage, and each section names the
    script that uses it. The logic that applies a setting lives in that script, not here.
"""
from dataclasses import dataclass, field
from pathlib import Path

# Project root: this file is tourism_project/config.py, so the root is two levels up.
# In Colab that is /content, locally it is the repository folder.
PROJECT_ROOT = Path(__file__).resolve().parents[1]


@dataclass(frozen=True)
class Config:
    """All pipeline settings. Frozen, so no script can change a setting by accident."""

    # =========================================================================
    # Paths and Hugging Face Hub (used by all scripts)
    # =========================================================================
    project_root: Path = PROJECT_ROOT

    hf_user: str = "richvrb"
    hf_dataset_name: str = "tourism-wellness-data"  # private: holds customer data

    # File names, identical locally and on the Hub
    raw_data_file: str = "tourism.csv"
    train_file: str = "train.csv"
    test_file: str = "test.csv"

    # =========================================================================
    # Columns (used by all scripts)
    # =========================================================================
    # What the model predicts: 1 = the customer bought a package, 0 = did not. About 19% of
    # the 4,128 customers in the dataset bought one.
    target: str = "ProdTaken"

    # Model features: customer profile information that is known BEFORE the sales team makes
    # contact. The prediction is used to decide whom to contact, so it has to work with what is
    # known at that moment.
    numeric_features: tuple[str, ...] = (
        "Age",
        "CityTier",
        "NumberOfPersonVisiting",
        "PreferredPropertyStar",
        "NumberOfTrips",
        "NumberOfChildrenVisiting",
        "MonthlyIncome",
    )
    binary_features: tuple[str, ...] = ("Passport", "OwnCar")
    categorical_features: tuple[str, ...] = (
        "TypeofContact",
        "Occupation",
        "Gender",
        "MaritalStatus",
        "Designation",
    )

    # Identifier: not a feature, only used to check that no customer is in train and test.
    id_column: str = "CustomerID"

    # Columns that describe the sales pitch and therefore only exist AFTER contact. A model
    # trained with them would look good on paper but could not be used before contact, when
    # these values are still unknown (data leakage). They are dropped in prep.py.
    post_contact_columns: tuple[str, ...] = (
        "DurationOfPitch",
        "NumberOfFollowups",
        "PitchSatisfactionScore",
        "ProductPitched",
    )

    # Group id written by prep.py. The dataset contains near-copies of customer records; rows
    # that are copies of each other share a group id, and a group is never split across train
    # and test (or across cross-validation folds), so the model is never evaluated on a copy of
    # a customer it was trained on. It is not a model feature.
    group_column: str = "group_id"

    # =========================================================================
    # Reproducibility (used by all scripts)
    # =========================================================================
    random_state: int = 42  # same seed everywhere, so every run gives the same result

    # =========================================================================
    # Data preparation (used by model_building/prep.py)
    # =========================================================================
    # Cleaning rules (step 2), based on an inspection of the raw data. They are fixed values,
    # not computed from the data, so the test rows cannot influence how training data is cleaned.

    # Gender contains "Fe Male" (155 rows), a spelling variant of "Female".
    gender_replacements: tuple[tuple[str, str], ...] = (("Fe Male", "Female"),)

    # In the raw data every monthly income lies between 16,009 and 38,304, except three values:
    # 1,000, 4,678 and 98,678. The wide limits below catch exactly those entry errors, which
    # become NaN and are filled in later by the model's imputer.
    monthly_income_range: tuple[float, float] = (5_000.0, 50_000.0)

    # Every customer takes 1-8 trips a year, except four values of 19-22: entry errors -> NaN.
    max_number_of_trips: int = 10

    # Finding near-copies (step 3). Each row pair gets an evidence score in bits for how unlikely
    # its agreement is by chance (maximum about 31). Across all 8.5 million pairs the scores form
    # two clusters with an almost empty gap between 23 and 27 bits (73 pairs); a threshold inside
    # that gap separates copies from ordinary similarity, so its exact value hardly matters.
    copy_threshold_bits: float = 24.0
    comparison_chunk_size: int = 256  # rows compared per batch, keeps memory use low

    # Train/test split (step 4). The test set is kept aside to estimate performance on new
    # customers; the checks guard against a split that would make that estimate misleading.
    test_size: float = 0.2              # 20% of the customers go to the test set
    max_prevalence_gap: float = 0.01    # share of buyers may differ at most 1 point, train vs test
    max_test_share_gap: float = 0.02    # realised test share may differ at most 2 points from 20%

    # =========================================================================
    # Model training (used by model_building/train.py)
    # =========================================================================
    # Every rule below was fixed BEFORE any model was trained. Choosing thresholds after seeing
    # the results would let the favourite model win by construction and make the selection
    # meaningless. The reasoning behind each rule is in the header comments of train.py.

    # --- Registration: the selected model goes to a private Hugging Face model repository ---
    hf_model_name: str = "tourism-wellness-model"
    model_file: str = "model.joblib"                  # fitted pipeline: preprocessing + model
    metadata_file: str = "model_metadata.json"        # everything the app needs to use the model
    operating_points_file: str = "operating_points.csv"

    # --- Experiment tracking (MLflow) ---
    mlflow_experiment: str = "tourism-wellness-precontact"
    feature_set_name: str = "pre_contact"  # tag on every run: which features the model may use
    max_tuning_runs: int = 5               # a hyperparameter search logs only its 5 best candidates

    # --- Validation ---
    # Models are compared with cross-validation on the train set only; the test set is used
    # once, at the very end. Folds are group-aware, like the train/test split, so a copy of a
    # customer never sits in the training folds while the original is being validated.
    cv_folds: int = 5
    # The two best models are compared once more on 3 different fold arrangements (15 folds in
    # total), because with ~640 buyers in train a single 5-fold run is too noisy to separate
    # models that differ by a few hundredths.
    finale_seeds: tuple[int, ...] = (42, 43, 44)

    # Models are ranked on PR-AUC (average precision): how well they put buyers above
    # non-buyers across the whole ranking. With only ~19% buyers it is more informative than
    # ROC-AUC, and unlike F1 it does not depend on a chosen probability threshold.
    scoring: str = "average_precision"
    search_n_iter: int = 30  # hyperparameter combinations tried per tuned model

    # --- Hyperparameter search spaces ("model__" addresses the model step inside the Pipeline) ---
    rf_search_space: dict = field(default_factory=lambda: {
        "model__n_estimators": [200, 400, 600, 800],
        "model__max_depth": [None, 6, 10, 16, 24],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__max_features": ["sqrt", "log2", 0.5],
        # Both options handle the 19/81 imbalance; which works better with bootstrapping
        # cannot be reasoned out in advance, so the search decides.
        "model__class_weight": ["balanced", "balanced_subsample"],
    })
    xgb_search_space: dict = field(default_factory=lambda: {
        "model__n_estimators": [200, 400, 600, 800],
        "model__max_depth": [3, 4, 6, 8, 10],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 3, 5, 10],
        "model__gamma": [0, 0.1, 0.5, 1.0],
        "model__reg_lambda": [1, 5, 10],
        # scale_pos_weight is NOT searched: it is fixed at (non-buyers / buyers). The imbalance
        # is a known fact, and the precision/recall trade-off is handled by the threshold.
    })

    # --- Selection rules ---
    # Gate: CV PR-AUC must beat "guess the base rate" (the dummy model) by at least 0.10.
    # There is deliberately no gate on how much PR-AUC varies between folds: with only ~130 buyers
    # per validation fold every good model varies by 0.06-0.09, and chance differences between
    # models are handled by the final comparison on 15 paired folds instead.
    min_lift_over_dummy: float = 0.10
    # How large the PR-AUC difference between the two finalists must be to count as real: this many
    # times the standard error of that difference over the 15 final folds. The standard error says
    # how much the average difference still wobbles, so 1.0 means "the lead must exceed the noise";
    # anything smaller is a tie and is decided by the tie-breaks in train.py.
    tie_tolerance_se: float = 1.0
    # Last tie-break: prefer the simpler model family (faster, easier to explain, lighter app).
    simplicity_order: tuple[str, ...] = ("random_forest", "xgboost")

    # --- Business view: marketing contacts a fixed budget of customers ---
    # The size of the customer base is unknown, so the budget is expressed as a share of it.
    # Precision ("share of contacted customers who buy") and lift ("times better than calling
    # at random") are reported for the top 5%, 10% and 20% of the model's ranking.
    contact_fractions: tuple[float, ...] = (0.05, 0.10, 0.20)
    # The share used as the first tie-break between the two finalists, and the one reported next to
    # the ranking metrics for every candidate. If PR-AUC cannot separate two models, the decision
    # falls on the part of the ranking that is actually called. 10% is the middle of the three
    # budgets above: 5% is too small to measure reliably, 20% dilutes the difference at the top.
    tiebreak_contact_fraction: float = 0.10
    n_bootstrap: int = 1_000                 # resamples for the uncertainty ranges on test

    # --- Final checks on the test set ---
    max_cv_test_gap: float = 0.05          # test PR-AUC far from the CV estimate -> warning
    leakage_roc_auc_ceiling: float = 0.90  # pre-contact features cannot plausibly score higher;
                                           # above it the script stops before registering

    # =========================================================================
    # Deployment and hosting (used by hosting/hosting.py)
    # =========================================================================
    # The Streamlit app runs in a public Hugging Face Space built from a Dockerfile. The app itself
    # (deployment/app.py) does not import this Config, because the container only contains the
    # files listed below; it reads everything it needs from model_metadata.json instead.
    hf_space_name: str = "tourism-wellness-app"
    space_app_port: int = 8501  # must equal app_port in deployment/README.md and EXPOSE in the Dockerfile

    # The only files ever uploaded to the Space: what the container needs plus the Space's own
    # configuration. A new file in deployment/ is therefore never published by accident. This
    # mirrors the whitelist in deployment/.dockerignore.
    deployment_files: tuple[str, ...] = (
        "Dockerfile",
        ".dockerignore",
        "README.md",               # Space configuration (sdk, app_port) in its front matter
        "requirements.txt",
        "app.py",
        ".streamlit/config.toml",
    )

    # Building the image and starting the app takes a few minutes; hosting.py checks the status
    # every 15 seconds and gives up after 20 minutes.
    space_build_timeout_s: int = 1_200
    space_poll_interval_s: int = 15

    # =========================================================================
    # Derived values (computed from the settings above, never set directly)
    # =========================================================================
    @property
    def hf_dataset_repo(self) -> str:
        """Full id of the dataset repository, e.g. 'richvrb/tourism-wellness-data'."""
        return f"{self.hf_user}/{self.hf_dataset_name}"

    @property
    def hf_model_repo(self) -> str:
        """Full id of the model repository, e.g. 'richvrb/tourism-wellness-model'."""
        return f"{self.hf_user}/{self.hf_model_name}"

    @property
    def hf_space_repo(self) -> str:
        """Full id of the Space, e.g. 'richvrb/tourism-wellness-app'."""
        return f"{self.hf_user}/{self.hf_space_name}"

    @property
    def space_url(self) -> str:
        """Public address of the running app, e.g. 'https://richvrb-tourism-wellness-app.hf.space'."""
        return f"https://{self.hf_user}-{self.hf_space_name}.hf.space"

    @property
    def deployment_dir(self) -> Path:
        """Folder with the files that make up the Space."""
        return self.project_root / "tourism_project" / "deployment"

    @property
    def model_dir(self) -> Path:
        """Folder where train.py saves the model bundle before uploading it."""
        return self.work_dir / "model"

    @property
    def mlflow_tracking_uri(self) -> str:
        """MLflow database location; MLflow 3 requires a database instead of a plain folder."""
        return f"sqlite:///{self.work_dir / 'mlflow.db'}"

    @property
    def data_path(self) -> Path:
        """Location of the raw CSV in the project, used when registering the dataset."""
        return self.project_root / "tourism_project" / "data" / self.raw_data_file

    @property
    def work_dir(self) -> Path:
        """Folder for generated files such as train.csv and test.csv; never committed."""
        return self.project_root / "outputs"

    @property
    def feature_columns(self) -> list[str]:
        """All model features in a fixed order."""
        return [*self.numeric_features, *self.binary_features, *self.categorical_features]

    @property
    def split_columns(self) -> list[str]:
        """Columns of train.csv and test.csv: features, target and group id, in this order."""
        return [*self.feature_columns, self.target, self.group_column]

Overwriting tourism_project/config.py


The next two cells write small helper modules that several stages share:

- `hub.py` creates private Hugging Face repositories and reads the prepared train and test files
  at an exact revision (used by data registration and model training).
- `ci.py` holds the few lines needed when a stage runs inside GitHub Actions: passing a value such
  as the data revision on to the next job, and adding result tables to the workflow's summary page.
  Outside GitHub Actions these functions do nothing, so the notebook is unaffected.

In [34]:
%%writefile tourism_project/hub.py
"""Shared helpers for the pipeline's Hugging Face Hub repositories.

Context
    The Hugging Face Hub is the shared storage between the pipeline stages: data registration
    and data preparation write the dataset repository, model training reads the prepared splits
    from it and writes the selected model to a model repository. The same helpers are used in
    the notebook and in GitHub Actions.

    A Hub repository keeps every upload as a commit with an id (a "revision"). Reading a file
    at a specific revision guarantees that training uses exactly the split that data
    preparation produced, even if the repository receives a newer commit later.

What this module offers
    - ensure_private_repo    : create a dataset or model repository if needed and keep it private
    - resolve_revision       : decide which revision (commit) of a repository to use
    - resolve_data_revision  : the dataset revision for training
    - resolve_model_revision : the model revision for the app
    - load_split             : download one split file at a revision and check its columns
"""
import os

import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

from tourism_project.config import Config


def ensure_private_repo(api: HfApi, repo_id: str, repo_type: str) -> None:
    """Create a Hub repository (dataset or model) if it does not exist, and make sure it is private."""
    # exist_ok=True: no error when the repository already exists, e.g. on every pipeline run
    # after the first one.
    api.create_repo(repo_id=repo_id, repo_type=repo_type, visibility="private", exist_ok=True)

    # create_repo ignores the visibility setting for a repository that already exists. Setting it
    # again explicitly guarantees the data or model stays private even if someone changed the
    # setting by hand on the website.
    api.update_repo_settings(repo_id=repo_id, repo_type=repo_type, visibility="private")


def resolve_revision(
    api: HfApi, repo_id: str, repo_type: str, env_variable: str, revision: str | None = None
) -> str:
    """Return the revision to use: the one passed in, the value of env_variable, or the newest commit."""
    # 1. A revision passed in explicitly always wins.
    if revision:
        return revision

    # 2. In GitHub Actions the previous job passes on the commit it produced (DATA_REVISION from data
    #    preparation, MODEL_REVISION from training), so the next job uses exactly that version.
    revision_from_pipeline = os.environ.get(env_variable)
    if revision_from_pipeline:
        return revision_from_pipeline

    # 3. Otherwise (for example in the notebook) use the newest commit of the repository.
    return api.repo_info(repo_id, repo_type=repo_type).sha


def resolve_data_revision(api: HfApi, cfg: Config, revision: str | None = None) -> str:
    """Dataset revision for training: explicit, $DATA_REVISION, or the newest dataset commit."""
    return resolve_revision(api, cfg.hf_dataset_repo, "dataset", "DATA_REVISION", revision)


def resolve_model_revision(api: HfApi, cfg: Config, revision: str | None = None) -> str:
    """Model revision for the app: explicit, $MODEL_REVISION, or the newest model commit."""
    return resolve_revision(api, cfg.hf_model_repo, "model", "MODEL_REVISION", revision)


def load_split(cfg: Config, filename: str, revision: str) -> pd.DataFrame:
    """Download one split file (train.csv or test.csv) at a fixed revision and read it."""
    local_path = hf_hub_download(
        cfg.hf_dataset_repo, filename, repo_type="dataset", revision=revision
    )
    split = pd.read_csv(local_path)

    # The model code relies on these exact columns in this order; stop with a clear message
    # if the file on the Hub was made by a different version of prep.py.
    if list(split.columns) != cfg.split_columns:
        raise ValueError(
            f"{filename} at revision {revision[:8]} has columns {list(split.columns)}, "
            f"expected {cfg.split_columns}"
        )
    return split

Overwriting tourism_project/hub.py


In [35]:
%%writefile tourism_project/ci.py
"""Helpers for running pipeline stages inside GitHub Actions.

Context
    In GitHub Actions every stage of the pipeline (data preparation, model training, ...) runs
    as a separate job on its own fresh machine. Jobs share nothing by default, so a stage that
    produces something the next stage needs, such as the id of the dataset version it uploaded,
    has to hand it over explicitly. GitHub also shows a summary page for every workflow run,
    which is the easiest place to read a stage's results.

    Both helpers do nothing when the code runs elsewhere (Colab, a local terminal), so the same
    scripts work unchanged in the notebook and in the pipeline.

What this module offers
    - export_github_output : pass a value (e.g. data_revision) on to later jobs
    - write_step_summary   : add markdown (e.g. a results table) to the workflow run's summary page
    - markdown_table       : format a DataFrame as a markdown table for that summary page
"""
import os

import pandas as pd


def export_github_output(name: str, value: str) -> None:
    """Pass a value to later jobs when running inside GitHub Actions; do nothing elsewhere."""
    # GitHub provides a file via GITHUB_OUTPUT; every "name=value" line written to it becomes an
    # output of the current step that later jobs can read. Outside GitHub Actions it is not set.
    output_file = os.environ.get("GITHUB_OUTPUT")
    if output_file:
        with open(output_file, "a", encoding="utf-8") as handle:
            handle.write(f"{name}={value}\n")


def write_step_summary(markdown: str) -> None:
    """Append markdown to the GitHub Actions run summary; do nothing elsewhere."""
    # GITHUB_STEP_SUMMARY points to a markdown file that GitHub renders on the run's summary page.
    summary_file = os.environ.get("GITHUB_STEP_SUMMARY")
    if summary_file:
        with open(summary_file, "a", encoding="utf-8") as handle:
            handle.write(markdown.rstrip("\n") + "\n\n")


def markdown_table(frame: pd.DataFrame, float_format: str = "{:.3f}") -> str:
    """Format a DataFrame (index included) as a markdown table; no extra packages needed."""
    # pandas' own to_markdown() needs the optional 'tabulate' package, which GitHub Actions
    # would have to install just for this; a plain loop keeps the pipeline's dependencies small.
    table = frame.reset_index()

    def cell(value: object) -> str:
        if isinstance(value, float):
            return "" if pd.isna(value) else float_format.format(value)
        return str(value)

    header = "| " + " | ".join(str(column) for column in table.columns) + " |"
    divider = "|" + "---|" * len(table.columns)
    rows = ["| " + " | ".join(cell(value) for value in row) + " |" for row in table.itertuples(index=False)]
    return "\n".join([header, divider, *rows])

Overwriting tourism_project/ci.py


## Data Registration

In [36]:
# === PHASE 03a: DATA FOLDER ===
# Create the folder for the raw dataset. In Colab, upload tourism.csv into this folder
# (Files panel on the left) before running the next cells.
os.makedirs("tourism_project/data", exist_ok=True)

# Show whether the dataset is in place, so a missing upload is noticed before registration.
raw_data_path = "tourism_project/data/tourism.csv"
if os.path.isfile(raw_data_path):
    print(f"Data folder ready, {raw_data_path} found ({os.path.getsize(raw_data_path):,} bytes)")
else:
    print(f"Data folder ready, but {raw_data_path} is missing: upload it before continuing")

Data folder ready, tourism_project/data/tourism.csv found (477,468 bytes)


Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

**Why register the data.** Every later stage, in Colab and in GitHub Actions, must start from the
same raw data. Registering places `tourism.csv` in one central, versioned location: a dataset
repository on the Hugging Face Hub. A Hub repository works like a Git repository, so every upload
is a commit with an id and each version of the data can be referred to later. The repository is
private, because the file contains customer data such as age, income and occupation.

The script below creates the repository if it does not exist yet, makes sure it is private, and
uploads `tourism.csv`. If the file on the Hub is already identical, no new commit is made. The
output shows the repository link, confirms it is private, lists its files and gives the commit id.

In [37]:
%%writefile tourism_project/model_building/data_register.py
"""Data registration stage of the Visit with Us MLOps pipeline.

Context
    The pipeline predicts, before the sales team contacts a customer, whether that customer is
    likely to buy the Wellness Tourism Package. It has four stages:

        data_register.py  ->  prep.py            ->  train.py        ->  deployment
        raw data on Hub       train/test on Hub      model on Hub        Streamlit app

    This script is the first stage. "Registering" the data means placing the raw file in one
    central, versioned location that every later stage reads from. That location is a dataset
    repository on the Hugging Face Hub. A Hub repository works like a Git repository: every
    upload is a commit with an id, so each version of the data can be referred to later.

    The repository is private because tourism.csv contains customer data (age, income,
    occupation, ...). Only people and pipelines with a Hugging Face token can read it.

What it does
    1. Repository - create the dataset repository if it does not exist yet, and make sure it
                    is private
    2. Upload     - upload tourism.csv from tourism_project/data/

How the file is organised
    Each step has a header with its input, output and reasoning, followed by one function.
    main() at the bottom connects the steps. Names and paths come from the central Config.

Running it
    python -m tourism_project.model_building.data_register   (from the project root)
    Runs the same way from the notebook (%run -m) and in GitHub Actions (python -m).
    Needs a Hugging Face token with write access, from HF_TOKEN or `hf auth login`.
    Running it again with an unchanged file creates no new commit on the Hub.
"""
from huggingface_hub import HfApi

from tourism_project.config import Config
from tourism_project.hub import ensure_private_repo


# =============================================================================
# Step 1 - Repository
# =============================================================================
#
# Input : the repository name from Config (richvrb/tourism-wellness-data)
# Output: a private dataset repository on the Hugging Face Hub, created if it is missing
#
# The first run creates the repository; every later run, for example each GitHub Actions run,
# finds it already exists and continues. Visibility is set explicitly on every run (see
# hub.ensure_private_repo), so the customer data stays private even if someone changed the
# setting by hand on the website.
#

def ensure_private_dataset_repo(api: HfApi, cfg: Config) -> None:
    """Create the dataset repository if needed, and make sure it is private."""
    # The shared helper is also used by model training for the model repository.
    ensure_private_repo(api, cfg.hf_dataset_repo, repo_type="dataset")


# =============================================================================
# Step 2 - Upload
# =============================================================================
#
# Input : tourism_project/data/tourism.csv (uploaded into Colab, or committed in the repository)
# Output: the same file in the dataset repository, plus the id of the commit
#
# After this step the Hub holds the registered raw data. Data preparation reads it from there,
# in the notebook as well as in GitHub Actions, so later stages never depend on a local file.
#

def upload_raw_data(api: HfApi, cfg: Config) -> str:
    """Upload tourism.csv and return the id of the resulting commit."""
    # A clear message is more helpful than the Hub's error when the file was not uploaded
    # to the data folder (in Colab this is a manual step).
    if not cfg.data_path.is_file():
        raise FileNotFoundError(f"Raw dataset not found at {cfg.data_path}")

    # Upload just this one file, not the whole data folder: only the raw CSV belongs in
    # the registration. If the file is unchanged, the Hub skips the commit.
    commit = api.upload_file(
        path_or_fileobj=cfg.data_path,
        path_in_repo=cfg.raw_data_file,
        repo_id=cfg.hf_dataset_repo,
        repo_type="dataset",
        commit_message=f"Register raw dataset {cfg.raw_data_file}",
    )
    return commit.oid


# =============================================================================
# Run all steps
# =============================================================================
#
# main() connects the steps and prints where the data ended up. It prints repository
# details only, never the contents of the file.
#

def main() -> None:
    """Run steps 1 and 2, then print where the data lives."""
    cfg = Config()
    api = HfApi()  # picks up the token from HF_TOKEN or `hf auth login`

    # 1. Repository
    ensure_private_dataset_repo(api, cfg)

    # 2. Upload (remember the latest commit first, to report whether anything changed)
    commit_before = api.repo_info(cfg.hf_dataset_repo, repo_type="dataset").sha
    commit_after = upload_raw_data(api, cfg)

    # Report
    info = api.repo_info(cfg.hf_dataset_repo, repo_type="dataset")
    files = sorted(sibling.rfilename for sibling in info.siblings)
    unchanged_note = "  (unchanged, no new commit)" if commit_after == commit_before else ""

    print(f"Dataset repo : https://huggingface.co/datasets/{cfg.hf_dataset_repo}")
    print(f"Private      : {info.private}")
    print(f"Files        : {', '.join(files)}")
    print(f"Commit       : {commit_after}{unchanged_note}")


if __name__ == "__main__":
    main()

Overwriting tourism_project/model_building/data_register.py


In [38]:
# === PHASE 03b: REGISTER DATASET ===
# Run the script written above: create the private dataset repository and upload tourism.csv.
# %run executes the script in this notebook's own Python, so an error stops the cell visibly
# (a failing `!python ...` command would not stop the notebook).
%run -m tourism_project.model_building.data_register

No files have been modified since last commit. Skipping to prevent empty commit.


Dataset repo : https://huggingface.co/datasets/richvrb/tourism-wellness-data
Private      : True
Files        : .gitattributes, test.csv, tourism.csv, train.csv
Commit       : 0d0e7fe27a22e462a9ec09eaa182d0c0d5e42892  (unchanged, no new commit)


## Data Preparation

**What data preparation does.** It turns the registered raw data into the train set that the model
learns from and the test set that is kept aside to estimate how well the model predicts *new*
customers. The cell below prints the counts behind every step, so the decisions can be checked against the data:
4,128 customers, about 19% of whom bought a package. Three decisions follow from that inspection:

1. **Cleaning.** `Gender` contains the spelling variant "Fe Male" (155 rows), which is merged into
   "Female". Three `MonthlyIncome` values lie far outside the range of all other incomes
   (16,009–38,304 after cleaning), and four `NumberOfTrips` values lie far above all others (1–8).
   These are entry errors and are set to missing; the model's imputer fills them in during training.
   Rows are kept, because the customer's other values are still useful.

2. **Near-copies.** Many rows are near-copies of other rows: the same customer profile with one or
   two values changed, most likely created when this teaching dataset was enlarged. The script
   finds them by scoring how unlikely it is that two rows agree by chance: an identical income to
   the unit is strong evidence, an identical yes/no value is weak evidence. Three quarters of the rows
   turn out to have at least one near-copy, and copies get a shared group id.

3. **Split.** The cell below makes an ordinary random split as a comparison and counts how many test
   rows end up with a near-copy in the train set: almost 60% of them. Their test score would partly
   measure recognition of known customers instead of prediction for new ones. The split therefore keeps
   each group of copies entirely in train or entirely in test, which leaves none. It is also stratified:
   with only about 19% buyers, a purely random test set could hold noticeably more or fewer buyers than
   the data as a whole.

The output then shows, per set, the number of rows, the share of buyers and the number of groups; it
confirms that no group appears in both sets; and it prints the `DATA_REVISION`, the commit id of
this exact version of train and test. In GitHub Actions that id is passed on to the training job,
so training always uses the split this run produced.

In [39]:
%%writefile tourism_project/model_building/prep.py
"""Data preparation stage of the Visit with Us MLOps pipeline.

Context
    Visit with Us wants to know, before the sales team contacts a customer, whether that
    customer is likely to buy the new Wellness Tourism Package (column ProdTaken, 1 = bought).
    Marketing can then spend its calls on the most promising customers. The pipeline that
    delivers this prediction has four stages, each building on the output of the previous one:

        data_register.py  ->  prep.py            ->  train.py        ->  deployment
        raw data on Hub       train/test on Hub      model on Hub        Streamlit app

    This script is the second stage. It turns the registered raw data into the train and test
    sets that model training learns from and is evaluated on. The same code runs in two places:
    from the notebook (%run -m) and as a job in GitHub Actions (python -m). Both read from and
    write to the Hugging Face Hub, so both produce identical files.

What it does
    1. Load  - read tourism.csv (4,128 customers, 20 columns) from the private dataset
               repository on the Hugging Face Hub
    2. Clean - merge a spelling variant and mark impossible values as missing
    3. Group - detect rows that are near-copies of each other and give them one group id
    4. Split - make a stratified 80/20 train/test split that never separates copies
    5. Save  - write train.csv and test.csv and upload both to the same repository

Why step 3 matters
    About three quarters of the rows have a near-copy elsewhere in the data: the same customer
    profile with one or two values changed, most likely created when this teaching dataset was
    enlarged. With an ordinary random split, 478 of the 826 test rows had a near-copy in the
    training set. A model evaluated on that test set would partly be recognising customers it
    has already seen, so its score would overstate how well it predicts new customers. Keeping
    each group of copies on one side of the split brings that number to 0.

How the file is organised
    Settings come from the central Config (config.py, section "Data preparation") and are
    passed to every function, so this file contains no hardcoded values. Each step has a header
    with its input, output and reasoning, followed by small functions that each do one task.
    main() at the bottom is the only place where the steps are connected.

Running it
    python -m tourism_project.model_building.prep   (from the project root)
    Needs a Hugging Face token with write access, from HF_TOKEN or `hf auth login`.
    Prints counts only, never customer rows, because notebook output and workflow logs are
    public. Running it twice gives identical files, and the Hub then creates no new commit.
"""
from collections.abc import Iterator
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

from tourism_project.ci import export_github_output
from tourism_project.config import Config


# =============================================================================
# Step 1 - Load
# =============================================================================
#
# Input : the private dataset repository on the Hugging Face Hub (filled by data_register.py)
# Output: a DataFrame with the 4,128 raw customer rows and all 20 columns
#
# The raw file is read from the Hub, not from tourism_project/data/. The Hub copy is the
# registered version of the data: GitHub Actions has no Colab upload folder, and reading the
# same registered file everywhere guarantees that the notebook and the pipeline start from
# identical data.
#
# The Hugging Face token (needed because the repository is private) is picked up automatically
# from the HF_TOKEN environment variable or from a local `hf auth login`.
#

def load_raw(cfg: Config) -> pd.DataFrame:
    """Download tourism.csv from the private dataset repository and read it into a DataFrame."""
    local_path = hf_hub_download(cfg.hf_dataset_repo, cfg.raw_data_file, repo_type="dataset")

    # index_col=0: the CSV starts with an unnamed row-number column that carries no information.
    return pd.read_csv(local_path, index_col=0)


# =============================================================================
# Step 2 - Clean
# =============================================================================
#
# Input : the raw DataFrame
# Output: a cleaned copy with the same 4,128 rows and 20 columns
#
# An inspection of the raw data found three problems. Each has a fixed rule in Config:
#   1. Gender has three values: "Male", "Female" and "Fe Male" (155 rows). "Fe Male" is a
#      spelling variant of "Female" and is merged into it.
#   2. MonthlyIncome: all incomes lie between 16,009 and 38,304, except three values
#      (1,000, 4,678 and 98,678). These are entry errors and are set to NaN.
#   3. NumberOfTrips: all values are 1-8, except four values of 19-22. Also entry errors,
#      also set to NaN.
#
# Why NaN instead of deleting the row: the customer's other 19 values are still useful.
# The model pipeline (training stage) fills missing values in with an imputer.
#
# Why fixed limits instead of limits computed from the data, such as percentiles: limits
# computed from all rows would be influenced by the test rows. Nothing about the test set may
# influence how the training data is prepared, otherwise the test score is no longer an honest
# estimate for new customers.
#

def clean_values(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Apply the cleaning rules from Config and return a cleaned copy; the input is unchanged."""
    cleaned = df.copy()

    # Merge spelling variants into one category, e.g. "Fe Male" -> "Female".
    cleaned["Gender"] = cleaned["Gender"].replace(dict(cfg.gender_replacements))

    # Incomes outside the realistic range are entry errors -> NaN.
    low_income, high_income = cfg.monthly_income_range
    income_out_of_range = ~cleaned["MonthlyIncome"].between(low_income, high_income)
    cleaned.loc[income_out_of_range, "MonthlyIncome"] = np.nan

    # An unrealistically high number of trips per year is an entry error -> NaN.
    too_many_trips = cleaned["NumberOfTrips"] > cfg.max_number_of_trips
    cleaned.loc[too_many_trips, "NumberOfTrips"] = np.nan

    return cleaned


# =============================================================================
# Step 3 - Group copies
# =============================================================================
#
# Input : the cleaned DataFrame
# Output: the same DataFrame with an extra column group_id
#
# What was found: 194 pairs of rows are identical on all 14 features, and many more rows match
# on all but one or two features. These are near-copies of the same customer profile, not
# different customers who happen to be similar. Two observations support that:
#   - Scoring all 8.5 million row pairs (see below) gives two clearly separated clusters:
#     millions of pairs with ordinary similarity, about 1,500 pairs close to the maximum score,
#     and almost nothing in between (73 pairs between 23 and 27 bits).
#   - The method below never looks at CustomerID, yet 99% of the pairs it links turn out to be
#     exactly the same distance apart in customer number: the trace of a program that copied
#     records and changed a value or two.
#
# How copies are detected: compare every pair of rows and give the pair an "evidence score"
# that says how unlikely it is that two different customers agree this much by chance.
# Not every agreement is equally convincing:
#   - Two random customers have the same Passport value (yes/no) about half of the time.
#     That agreement proves almost nothing.
#   - Two random customers have exactly the same MonthlyIncome about once in 2,000 pairs.
#     That agreement proves a lot.
# So each feature gets a weight in bits: weight = -log2(chance that two random rows agree).
#   chance 0.5     -> 1 bit       (Passport)
#   chance 0.0005  -> ~11 bits    (MonthlyIncome)
# The score of a pair is the sum of the weights of the features on which both rows agree
# (maximum about 31 bits). Pairs scoring at least Config.copy_threshold_bits (24, in the
# empty gap between the two clusters) are copies. This is the standard idea behind record
# linkage: matching records that describe the same entity.
#
# Rows linked by copies, directly or through a chain (A~B and B~C), form one group. Result:
# 2,574 groups of 1, 2 or 4 rows; about three quarters of all rows belong to a group of 2+.
#
# The target (ProdTaken) is not compared: a copy leaks information even if its outcome differs.
#

def encode_features(df: pd.DataFrame, cfg: Config) -> np.ndarray:
    """Turn every feature value into an integer code, so rows can be compared quickly."""
    encoded_columns = []
    for column in cfg.feature_columns:
        # Convert to text first so that 3 and 3.0 get the same code, and give missing values
        # their own code so that two NaN values count as agreeing.
        as_text = df[column].astype("string").fillna("<NA>")
        codes, _unique_values = pd.factorize(as_text)
        encoded_columns.append(codes)

    # Result: one row per customer, one column per feature, small integers only.
    return np.column_stack(encoded_columns).astype(np.int32)


def agreement_weights(codes: np.ndarray) -> np.ndarray:
    """Weight per feature in bits: -log2 of the chance that two random rows agree on it."""
    weights = []
    for column_codes in codes.T:
        # Share of rows per value, e.g. Passport: 0.7 "no" and 0.3 "yes".
        value_shares = np.bincount(column_codes) / len(column_codes)

        # Chance that two randomly drawn rows have the same value = sum of the squared shares,
        # e.g. 0.7^2 + 0.3^2 = 0.58. Rare values push this chance down and the weight up.
        chance_of_agreement = np.sum(value_shares**2)

        weights.append(-np.log2(chance_of_agreement))
    return np.array(weights)


def iter_pair_scores(
    codes: np.ndarray, weights: np.ndarray, chunk_size: int
) -> Iterator[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """Yield (i, j, score) arrays for every row pair with i < j, one row i at a time."""
    n_rows = len(codes)

    # 4,128 rows give about 8.5 million pairs. A full rows x rows x features comparison would
    # need gigabytes of memory, so `chunk_size` rows are compared with all rows at a time.
    for chunk_start in range(0, n_rows, chunk_size):
        chunk_end = min(chunk_start + chunk_size, n_rows)
        chunk_codes = codes[chunk_start:chunk_end]

        # scores[a, b] = evidence that row a of this chunk and row b of the dataset are copies.
        scores = np.zeros((chunk_end - chunk_start, n_rows))
        for column_index, column_weight in enumerate(weights):
            # True where both rows have the same value for this feature; add its weight there.
            agrees = chunk_codes[:, [column_index]] == codes[:, column_index]
            scores += agrees * column_weight

        # Report every pair once and never compare a row with itself: only pairs with i < j.
        for position_in_chunk in range(chunk_end - chunk_start):
            i = chunk_start + position_in_chunk
            later_rows = np.arange(i + 1, n_rows)
            if len(later_rows) == 0:
                continue
            yield np.full(len(later_rows), i), later_rows, scores[position_in_chunk, i + 1:]


def find_copy_pairs(df: pd.DataFrame, cfg: Config) -> np.ndarray:
    """Return all row pairs whose evidence score reaches the copy threshold, as [i, j] rows."""
    codes = encode_features(df, cfg)
    weights = agreement_weights(codes)

    copy_pairs = []
    for rows_i, rows_j, scores in iter_pair_scores(codes, weights, cfg.comparison_chunk_size):
        is_copy = scores >= cfg.copy_threshold_bits
        if is_copy.any():
            copy_pairs.append(np.column_stack([rows_i[is_copy], rows_j[is_copy]]))

    if not copy_pairs:
        return np.empty((0, 2), dtype=int)
    return np.vstack(copy_pairs)


def add_group_id(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Add a group id column: rows that are copies, directly or through a chain, share it."""
    grouped = df.copy()
    copy_pairs = find_copy_pairs(grouped, cfg)
    n_rows = len(grouped)

    # Picture every row as a dot and every copy pair as a line between two dots. A group is a
    # set of dots connected by lines (a "connected component" in graph terms); a row without
    # any copy is a group on its own. scipy expects the lines as a sparse rows x rows matrix.
    line_matrix = coo_matrix(
        (np.ones(len(copy_pairs)), (copy_pairs[:, 0], copy_pairs[:, 1])),
        shape=(n_rows, n_rows),
    )
    _n_groups, group_per_row = connected_components(line_matrix, directed=False)

    grouped[cfg.group_column] = group_per_row
    return grouped


# =============================================================================
# Step 4 - Split
# =============================================================================
#
# Input : the grouped DataFrame (every row has a group_id)
# Output: a train set (about 80%, used to fit the model) and a test set (about 20%, used
#         only once at the end to estimate how well the model predicts new customers)
#
# The split has to meet two requirements at the same time:
#   - Groups stay together: all copies of a record go to train, or all go to test. This is
#     what keeps copies of training customers out of the test set (step 3).
#   - Stratified: train and test get about the same share of buyers. Only about 19% of the
#     customers bought a package, so a purely random test set of 826 rows could by chance
#     hold noticeably more or fewer buyers and give a misleading score.
# StratifiedGroupKFold from scikit-learn meets both requirements.
#
# check_split verifies the result and stops the script if a requirement is not met, so a
# broken split can never reach model training. select_columns then drops CustomerID and the
# post-contact columns, which the model must not see (see Config.post_contact_columns).
#

def split_by_group(df: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split into train and test so that each group lands entirely on one side."""
    # StratifiedGroupKFold divides the groups into k parts, each with about the same share of
    # buyers. With test_size 0.2 that is 5 parts; the first part becomes the test set and the
    # other four together form the train set.
    n_parts = round(1 / cfg.test_size)
    splitter = StratifiedGroupKFold(
        n_splits=n_parts, shuffle=True, random_state=cfg.random_state
    )
    parts = splitter.split(df, df[cfg.target], groups=df[cfg.group_column])
    train_positions, test_positions = next(parts)

    train = df.iloc[train_positions].copy()
    test = df.iloc[test_positions].copy()
    return train, test


def check_split(
    full: pd.DataFrame,
    train: pd.DataFrame,
    test: pd.DataFrame,
    cfg: Config,
) -> None:
    """Raise an error if the split lost rows, leaked a customer or group, or is unbalanced."""
    prevalence_gap = abs(train[cfg.target].mean() - test[cfg.target].mean())
    test_share_gap = abs(len(test) / len(full) - cfg.test_size)

    # Each check has a readable name, so a failure message says exactly which requirement broke.
    checks = {
        "no rows lost": len(train) + len(test) == len(full),
        "no customer in both sets": not set(train[cfg.id_column]) & set(test[cfg.id_column]),
        "no group in both sets": not set(train[cfg.group_column]) & set(test[cfg.group_column]),
        "share of buyers similar in train and test": prevalence_gap <= cfg.max_prevalence_gap,
        "test set has the intended size": test_share_gap <= cfg.max_test_share_gap,
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        # A real error instead of `assert`, because Python skips asserts when run with -O.
        raise ValueError(f"Split checks failed: {failed}")


def select_columns(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Keep only the features, the target and the group id, in a fixed column order."""
    # Dropped: CustomerID (an identifier, not information about the customer) and the four
    # post-contact columns. group_id stays because training needs it to keep copies together
    # in cross-validation as well; it is never used as a model feature.
    return df[cfg.split_columns].reset_index(drop=True)


# =============================================================================
# Step 5 - Save and upload
# =============================================================================
#
# Input : the train and test DataFrames
# Output: train.csv and test.csv in outputs/data/ and in the dataset repository on the Hub
#
# The files go to the Hub because the next stage, model training, reads them from there, both
# in the notebook and in GitHub Actions. That way every stage works with exactly the same data.
#
# Every upload to a Hugging Face repository is a commit with a unique id, just like in Git.
# That id is the DATA_REVISION: a permanent label for this exact version of train and test.
# In GitHub Actions it is handed to the training job, so training uses the split this run
# produced even if the repository receives another commit in the meantime.
#

def save_splits(cfg: Config, train: pd.DataFrame, test: pd.DataFrame) -> list[Path]:
    """Save train.csv and test.csv in the work folder (outputs/data/) and return their paths."""
    output_dir = cfg.work_dir / "data"
    output_dir.mkdir(parents=True, exist_ok=True)

    train_path = output_dir / cfg.train_file
    test_path = output_dir / cfg.test_file
    train.to_csv(train_path, index=False)
    test.to_csv(test_path, index=False)

    return [train_path, test_path]


def upload_splits(api: HfApi, cfg: Config, paths: list[Path]) -> str:
    """Upload train.csv and test.csv in one commit and return that commit's id."""
    # One commit for both files, so every version of the repository holds a train set and a
    # test set that belong together. If both files are unchanged, the Hub skips the commit and
    # returns the id of the latest existing commit.
    files_to_upload = [
        CommitOperationAdd(path_in_repo=path.name, path_or_fileobj=path) for path in paths
    ]
    commit = api.create_commit(
        repo_id=cfg.hf_dataset_repo,
        repo_type="dataset",
        operations=files_to_upload,
        commit_message="Prepare cleaned train/test split",
    )
    return commit.oid


def plain_split_leakage(grouped: pd.DataFrame, cfg: Config) -> tuple[int, int]:
    """How many test rows an ordinary random split would leave with a near-copy in train."""
    # The comparison that justifies the group-aware split: split the same rows the ordinary way
    # (stratified on the target, ignoring the groups) and count the test rows whose group also
    # occurs in train. Those rows would be scored on a customer the model had already seen.
    train, test = train_test_split(
        grouped,
        test_size=cfg.test_size,
        stratify=grouped[cfg.target],
        random_state=cfg.random_state,
    )
    groups_in_train = set(train[cfg.group_column])
    leaking = int(test[cfg.group_column].isin(groups_in_train).sum())
    return leaking, len(test)


def print_data_quality(
    raw: pd.DataFrame, cleaned: pd.DataFrame, grouped: pd.DataFrame, cfg: Config
) -> None:
    """Print the counts behind the cleaning rules and the split, so no number is unexplained."""
    # Only counts and shares are printed: this output ends up in a public repository and in the
    # GitHub Actions log, where customer rows do not belong.
    print(f"Raw data       : {len(raw)} customers | buyers {raw[cfg.target].mean():.2%}")

    renamed = int((raw["Gender"] != cleaned["Gender"]).sum())
    incomes_removed = int(cleaned["MonthlyIncome"].isna().sum() - raw["MonthlyIncome"].isna().sum())
    trips_removed = int(cleaned["NumberOfTrips"].isna().sum() - raw["NumberOfTrips"].isna().sum())
    valid_income = cleaned["MonthlyIncome"].dropna()
    print(f"Cleaning       : {renamed} Gender spelling variants merged "
          f"| {incomes_removed} incomes and {trips_removed} trip counts outside the realistic range "
          f"-> missing")
    print(f"                 remaining incomes run from {valid_income.min():,.0f} to {valid_income.max():,.0f}")

    group_sizes = grouped[cfg.group_column].value_counts()
    rows_with_copy = int(group_sizes[group_sizes > 1].sum())
    print(f"Near-copies    : {rows_with_copy} of {len(grouped)} rows ({rows_with_copy / len(grouped):.0%}) "
          f"have at least one near-copy | {len(group_sizes)} groups")

    leaking, test_rows = plain_split_leakage(grouped, cfg)
    print(f"Why grouping   : an ordinary random split would leave {leaking} of {test_rows} test rows "
          f"with a near-copy in train; the group-aware split below leaves 0")


def print_report(
    cfg: Config, train: pd.DataFrame, test: pd.DataFrame, data_revision: str, unchanged: bool
) -> None:
    """Print a short summary with counts only, so no customer data appears in public output."""
    for name, split in (("Train", train), ("Test", test)):
        print(
            f"{name:<13} : {len(split)} rows "
            f"| buyers {split[cfg.target].mean():.2%} "
            f"| {split[cfg.group_column].nunique()} groups"
        )
    groups_in_both = set(train[cfg.group_column]) & set(test[cfg.group_column])
    print(f"Groups in both: {len(groups_in_both)}")
    unchanged_note = "  (unchanged, no new commit)" if unchanged else ""
    print(f"DATA_REVISION : {data_revision}{unchanged_note}")


# =============================================================================
# Run all steps
# =============================================================================
#
# main() is the only place where the steps are connected. Every function above receives what
# it needs as arguments and returns its result, so each step can also be run or inspected on
# its own, for example in a notebook cell.
#

def main() -> None:
    """Run steps 1 to 5 in order."""
    cfg = Config()
    api = HfApi()  # picks up the token from HF_TOKEN or `hf auth login`

    # 1. Load the registered raw data
    raw = load_raw(cfg)

    # 2. Clean
    cleaned = clean_values(raw, cfg)

    # 3. Group near-copies, then report the counts behind the cleaning rules and the grouping
    grouped = add_group_id(cleaned, cfg)
    print_data_quality(raw, cleaned, grouped, cfg)

    # 4. Split, check the split while CustomerID is still present, then drop extra columns
    train, test = split_by_group(grouped, cfg)
    check_split(grouped, train, test, cfg)
    train = select_columns(train, cfg)
    test = select_columns(test, cfg)

    # 5. Save locally, upload to the Hub, report
    paths = save_splits(cfg, train, test)
    commit_before = api.repo_info(cfg.hf_dataset_repo, repo_type="dataset").sha
    data_revision = upload_splits(api, cfg, paths)

    print_report(cfg, train, test, data_revision, unchanged=data_revision == commit_before)

    # In GitHub Actions: tell the training job which version of the data to use.
    export_github_output("data_revision", data_revision)


if __name__ == "__main__":
    main()

Overwriting tourism_project/model_building/prep.py


In [40]:
# === PHASE 04: RUN DATA PREPARATION ===
# Run the script written above: clean the data, group copies, split into train and test,
# and upload both files to the dataset repository. %run stops the cell on an error.
%run -m tourism_project.model_building.prep

Raw data       : 4128 customers | buyers 19.31%
Cleaning       : 155 Gender spelling variants merged | 3 incomes and 4 trip counts outside the realistic range -> missing
                 remaining incomes run from 16,009 to 38,304
Near-copies    : 3094 of 4128 rows (75%) have at least one near-copy | 2574 groups
Why grouping   : an ordinary random split would leave 479 of 826 test rows with a near-copy in train; the group-aware split below leaves 0


No files have been modified since last commit. Skipping to prevent empty commit.


Train         : 3302 rows | buyers 19.29% | 2059 groups
Test          : 826 rows | buyers 19.37% | 515 groups
Groups in both: 0
DATA_REVISION : 0d0e7fe27a22e462a9ec09eaa182d0c0d5e42892  (unchanged, no new commit)


## Model Training and Registration with Experimentation Tracking

**What model training does.** It turns the prepared train set into the model that marketing
uses to decide whom to contact, chooses that model with rules fixed before any model was trained,
measures how well it predicts new customers, and registers it for the app. Every step is recorded
in MLflow, so the choice can be traced afterwards.

The *Approach* section explained why the model must produce a ranking, why PR-AUC decides and why the
threshold is a separate, budget-driven choice. This section turns that into the procedure below.

**Candidates.** Five pipelines share the same preprocessing (missing values filled in, categories
one-hot encoded) and differ in the model:

| Run | Model | Role |
|---|---|---|
| `dummy_baseline` | always predicts the base rate of buyers | the floor every real model must beat |
| `randomforest_baseline` / `xgboost_baseline` | random forest / XGBoost, default settings | does tuning add anything? |
| `randomforest_tuned` / `xgboost_tuned` | the same, after trying 30 hyperparameter combinations | candidates with searched settings |

Buyers are only ~19% of the customers, so the models weight buyers more heavily
(`class_weight="balanced"`, and for XGBoost `scale_pos_weight` = non-buyers per buyer).

**How candidates are compared.** Every candidate is measured with 5-fold cross-validation on the
train set only. The result is one table with a row per candidate: its tuned hyperparameters next to
its cross-validated PR-AUC and ROC-AUC, and the precision, recall and F1 it reaches when the top 10% of
the ranking is contacted. The folds are group-aware, like the train/test split, so a near-copy of a customer
is never in the training folds while the original is being validated. The deciding metric is
PR-AUC (average precision): how well buyers are ranked above non-buyers. With few buyers it is
more informative than ROC-AUC, and unlike F1 it does not depend on a chosen threshold.

**What the top 10% columns add.** PR-AUC judges the whole ranking, which is what the choice should rest
on, but it is hard to picture. Precision, recall and F1 at the top 10% describe one concrete decision — call
the best-scoring tenth of the customers — and make the same quality readable: "of the customers called, this
share buys". They stay next to PR-AUC for three reasons: they show whether a higher PR-AUC also improves the
part of the ranking that is actually called, rather than the middle of the list; they are the first tie-break
when two finalists cannot be separated; and they connect the model choice to the thresholds, the test results
and the app, which all report the same numbers. They do not decide on their own: a single fold's top 10% holds
only about 66 customers, so a few buyers more or less move the figure by several points.

**How one model is chosen.**
1. *Gate* — a candidate must beat the dummy's PR-AUC by at least 0.10. Without such a floor the
   "best" model could simply be the least bad one.
2. *Ranking* — the two candidates with the highest cross-validated PR-AUC become finalists.
3. *Final* — the two finalists usually differ by only a few thousandths of PR-AUC, which is the
   order of magnitude of measurement noise: cross-validating the same model on a different split of
   the data already moves its score. Picking the winner on such a difference would be close to
   flipping a coin, so the final step tests whether the difference is real. Both finalists are
   cross-validated on the same 15 folds (5 folds × 3 seeds) and the difference in PR-AUC is taken
   *per fold*, which cancels out how easy or hard each fold happens to be. Those 15 differences give
   an average and a standard error — the standard error being the measure of how much that average
   still wobbles: if the two models swap places from fold to fold it is large, if the same model wins
   by a similar margin every time it is small.
   - Average difference larger than one standard error → the lead is bigger than the noise, and the
     better model wins.
   - Otherwise it counts as a tie, and the winner is decided by rules fixed in advance, in this
     order: precision in the top 10% (the decision marketing actually makes), then the smaller spread
     across folds, then the smaller gap between training and validation score, then the simpler model
     family.

   Everything here is on the PR-AUC scale of 0 to 1, so these numbers are small: a difference and a
   standard error of a few thousandths are normal. The output below states the difference, the standard
   error and which rule decided.

**Operating points.** A score becomes a contact decision through a threshold. Thresholds are set on
out-of-fold scores (every training customer scored by a model that did not see them) for four
situations: the best balance of precision and recall (`max_f1`), and contacting the top 5%, 10% or
20% of customers when marketing has a fixed calling budget. Lift says how many times better than
calling at random each option is.

**The test set is used once**, after the model and thresholds are fixed, so its result is an honest
estimate for new customers. Because the top 10% of the test set is only about 83 customers, precision
and lift are shown with 95% ranges. Two checks follow: the test PR-AUC should be within 0.05 of the
cross-validated estimate (otherwise a warning), and a test ROC-AUC above 0.90 stops the script before
registration, because profile data cannot plausibly predict that well without leaked information.

**Registration and tracking.** The model, `model_metadata.json` (package versions, feature lists,
known category values, thresholds and a few synthetic check examples for the app) and
`operating_points.csv` are uploaded in one commit to the private model repository
`richvrb/tourism-wellness-model`; that commit id is the `MODEL_REVISION` the app will load. MLflow holds
one run per candidate with all hyperparameters, cross-validated metrics and its selection outcome
(tag `selection_status`), the best combinations of each hyperparameter search as child runs, and a
final run with the test results.

**What the output shows:** progress per candidate, the candidates table (hyperparameters next to
metrics), the selection decision, the operating points on train and on test, the test ranges and
decisions, and the registered model. The run takes several minutes, most of it spent on the two
hyperparameter searches.

In [41]:
%%writefile tourism_project/model_building/train.py
"""Model training stage of the Visit with Us MLOps pipeline.

Context
    Visit with Us wants to know, before the sales team contacts a customer, whether that customer
    is likely to buy the Wellness Tourism Package (column ProdTaken, 1 = bought). Marketing can
    then contact the most promising customers first. The pipeline has four stages:

        data_register.py  ->  prep.py            ->  train.py        ->  deployment
        raw data on Hub       train/test on Hub      model on Hub        Streamlit app

    This script is the third stage. It compares several models, picks one with rules that were
    fixed before any model was trained, measures how well it predicts new customers, and
    registers it in a private Hugging Face model repository for the app to use. Every model and
    every tuning attempt is recorded in MLflow, so the choice can be traced afterwards. The same
    code runs from the notebook (%run -m) and as a job in GitHub Actions (python -m).

What it does
    1. Load train      - read train.csv from the Hub (test.csv stays untouched until step 6)
    2. Build models    - five candidate pipelines: a "guess the base rate" floor, and random
                         forest and XGBoost models with default and with tuned settings
    3. Evaluate        - tune where needed and cross-validate every candidate; log all
                         hyperparameters and metrics to MLflow
    4. Select          - a minimum-quality gate, ranking and a head-to-head final between
                         the two best models
    5. Operating points- turn model scores into contact decisions: best-F1 threshold and the
                         thresholds for contacting the top 5%, 10% or 20% of customers
    6. Test            - one final, honest measurement on the untouched test set
    7. Register        - upload the model, its metadata and the operating points to the Hub

Why these choices
    - Ranking quality, measured with PR-AUC (average precision), decides which model wins. With
      only ~19% buyers, what matters is whether buyers end up at the top of the list; PR-AUC
      measures exactly that and, unlike F1, does not depend on a chosen probability threshold.
    - Cross-validation folds are group-aware: rows that are near-copies of each other (see
      prep.py) always stay in the same fold, so a model is never validated on a copy of a
      customer it was trained on.
    - The test set is used exactly once, after the model has been chosen. Choosing with the test
      set would turn it into another validation set, and no honest score would be left.

How the file is organised
    Settings come from the central Config (config.py, section "Model training") and are passed to
    every function. Each step has a header with its input, output and reasoning, followed by
    small functions that each do one task. main() at the bottom connects the steps.

Running it
    python -m tourism_project.model_building.train   (from the project root)
    Needs a Hugging Face token with write access, from HF_TOKEN or `hf auth login`.
    Takes several minutes: two hyperparameter searches of 30 combinations x 5 folds each.
    Prints aggregate results only, never customer rows, because notebook output and workflow
    logs are public.
"""
import json
import os
import platform
from dataclasses import dataclass, field
from importlib.metadata import version

# MLflow prints a notice meant for AI coding assistants on import; it is not useful here.
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from huggingface_hub import CommitOperationAdd, HfApi
from huggingface_hub.utils import disable_progress_bars
from mlflow.tracking import MlflowClient
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from tourism_project.ci import export_github_output, markdown_table, write_step_summary
from tourism_project.config import Config
from tourism_project.hub import ensure_private_repo, load_split, resolve_data_revision


# =============================================================================
# Data containers
# =============================================================================
#
# Two small dataclasses keep related values together as they move through the steps:
# a Candidate describes a model before training, a CandidateResult holds what was measured.
#

@dataclass
class Candidate:
    """One model on the ladder: its name, role, untrained pipeline and optional search space."""

    name: str            # MLflow run name, e.g. "xgboost_tuned"
    family: str          # "dummy" | "random_forest" | "xgboost"
    stage: str           # "baseline" (default settings) or "tuned" (hyperparameter search)
    pipeline: Pipeline   # preprocessing + model, not yet trained
    search_space: dict | None = None


@dataclass
class CandidateResult:
    """Everything measured for one candidate, plus its selection outcome."""

    candidate: Candidate
    pipeline: Pipeline             # with the best hyperparameters for tuned candidates
    model_params: dict             # hyperparameters of the model step, as logged
    metrics: dict                  # cv_* and train_* metrics, as logged
    run_id: str                    # MLflow run that holds the details
    status: str = "candidate"      # reference | rejected_gate | ranked_out | finalist | selected
    reason: str = ""               # why a candidate was rejected or ranked out
    finale: dict = field(default_factory=dict)  # finale_* metrics for the two finalists


# =============================================================================
# Step 1 - Load train
# =============================================================================
#
# Input : the dataset repository on the Hub, at DATA_REVISION (set by the data preparation job
#         in GitHub Actions) or at the newest commit when run from the notebook
# Output: features X, target y and the group id of every training row, plus the revision used
#
# Only train.csv is loaded here. test.csv is read for the first time in step 6, after the model
# has been chosen, so nothing about the test set can influence any decision before that.
#

def split_columns(frame: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Separate a split file into features, target and group id."""
    return frame[cfg.feature_columns], frame[cfg.target], frame[cfg.group_column]


def load_training_data(
    api: HfApi, cfg: Config
) -> tuple[pd.DataFrame, pd.Series, pd.Series, str]:
    """Load train.csv from the Hub; return features, target, groups and the data revision."""
    data_revision = resolve_data_revision(api, cfg)
    train = load_split(cfg, cfg.train_file, data_revision)
    features, target, groups = split_columns(train, cfg)
    return features, target, groups, data_revision


# =============================================================================
# Step 2 - Build models
# =============================================================================
#
# Input : the training target (to measure the class imbalance)
# Output: five candidates, each a Pipeline of the same preprocessing followed by a model
#
# Preprocessing is part of every pipeline, so it is fitted inside each cross-validation fold on
# that fold's training rows only. Fitting it once on all training rows would let the validation
# rows influence, for example, the median used to fill in missing incomes.
#   - numeric features    : fill missing values (the 7 cleaned entry errors) with the median
#   - binary features     : fill missing values with the most frequent value (none occur today,
#                           but the app may receive incomplete input)
#   - categorical features: fill missing values with the most frequent value, then one-hot
#                           encode; categories never seen in training are ignored instead of
#                           causing an error
#
# The candidate ladder, from simple to complex:
#   dummy_baseline         always predicts the share of buyers; the floor every model must beat
#   randomforest_baseline  random forest with default settings
#   xgboost_baseline       gradient boosting with default settings
#   randomforest_tuned     random forest with searched hyperparameters
#   xgboost_tuned          gradient boosting with searched hyperparameters
# Every model except the dummy is a candidate; a baseline may win if tuning does not help.
#
# Class imbalance (about 81% non-buyers) is handled by weighting buyers more heavily:
# class_weight="balanced" for the tree models, scale_pos_weight = non-buyers / buyers for
# XGBoost. That weight is fixed, not searched: the imbalance is a known fact, and the trade-off
# between precision and recall is set later with the decision threshold (step 5).
#

def build_preprocessor(cfg: Config) -> ColumnTransformer:
    """Fill in missing values per feature type and one-hot encode the categorical features."""
    categorical_steps = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", SimpleImputer(strategy="median"), list(cfg.numeric_features)),
            ("binary", SimpleImputer(strategy="most_frequent"), list(cfg.binary_features)),
            ("categorical", categorical_steps, list(cfg.categorical_features)),
        ]
    )


def build_pipeline(model: object, cfg: Config) -> Pipeline:
    """Put the shared preprocessing in front of a model; the step names are used in search spaces."""
    return Pipeline([("preprocess", build_preprocessor(cfg)), ("model", model)])


def model_ladder(target: pd.Series, cfg: Config) -> list[Candidate]:
    """Return the five candidates, from the base-rate floor to the tuned models."""
    # Weight for buyers in XGBoost: how many non-buyers there are per buyer (about 4.2).
    buyer_weight = float((target == 0).sum() / (target == 1).sum())
    seed = cfg.random_state

    def xgboost() -> XGBClassifier:
        return XGBClassifier(
            scale_pos_weight=buyer_weight,
            eval_metric="logloss",
            tree_method="hist",
            random_state=seed,
            n_jobs=-1,
        )

    def random_forest() -> RandomForestClassifier:
        return RandomForestClassifier(class_weight="balanced", random_state=seed, n_jobs=-1)

    return [
        Candidate("dummy_baseline", "dummy", "baseline",
                  build_pipeline(DummyClassifier(strategy="prior"), cfg)),
        Candidate("randomforest_baseline", "random_forest", "baseline",
                  build_pipeline(random_forest(), cfg)),
        Candidate("xgboost_baseline", "xgboost", "baseline",
                  build_pipeline(xgboost(), cfg)),
        Candidate("randomforest_tuned", "random_forest", "tuned",
                  build_pipeline(random_forest(), cfg), cfg.rf_search_space),
        Candidate("xgboost_tuned", "xgboost", "tuned",
                  build_pipeline(xgboost(), cfg), cfg.xgb_search_space),
    ]


# =============================================================================
# Step 3 - Evaluate candidates
# =============================================================================
#
# Input : the five candidates and the training data
# Output: one CandidateResult per candidate, each logged as its own MLflow run
#
# All candidates are measured on exactly the same group-aware folds (one shared
# StratifiedGroupKFold): 5 folds, each fold keeping every group of copies together and holding
# about the same share of buyers. Only then is a difference between two models a difference
# between the models, and not between the folds they happened to get.
#
# For every fold the same metrics are computed by one function, ranking_metrics:
#   pr_auc               how well buyers are ranked above non-buyers (the deciding metric)
#   roc_auc              the same idea on a scale where 0.5 means random; for reference
#   top_{q}pct_precision share of buyers among the top q% of the ranking (q = 5, 10, 20)
#   top_{q}pct_lift      that share divided by the base rate: "q% contacted, x times better
#                        than calling at random"
# The fold values are summarised as mean and standard deviation (cv_pr_auc_mean, ...).
#
# Tuned candidates first run a RandomizedSearchCV: 30 random hyperparameter combinations, each
# scored by PR-AUC on the same 5 group-aware folds. MLflow autologging is switched on only during
# that search and records the 5 best combinations as child runs of the candidate's run, so all
# tried hyperparameters stay traceable. The best combination is then measured with the same
# cross-validation function as every other candidate, so all candidates are compared alike.
#
# train_pr_auc is the score on the training rows the model was fitted on. The gap to the CV score
# (overfit_gap_pr_auc) shows how much a model memorises instead of generalises. It is logged and
# used as a tie-break, but it is not a gate: random forests always score almost perfectly on
# their own training rows, which is how they work, not a defect.
#
# Candidate runs never receive test metrics. If test scores of all models stood side by side, the
# choice would effectively be made on the test set.
#

def share_key(share: float) -> str:
    """Name of a contact share in metric keys, e.g. 0.10 -> 'top_10pct'."""
    return f"top_{round(share * 100)}pct"


def top_share_mask(scores: np.ndarray, share: float) -> np.ndarray:
    """True for the highest-scoring `share` of rows: the customers marketing would contact."""
    n_contacted = max(1, round(share * len(scores)))
    ranking = np.argsort(-scores, kind="stable")  # highest score first; ties keep row order
    mask = np.zeros(len(scores), dtype=bool)
    mask[ranking[:n_contacted]] = True
    return mask


def ranking_metrics(target: pd.Series, scores: np.ndarray, cfg: Config) -> dict:
    """PR-AUC, ROC-AUC and precision, recall, F1 and lift per contact share: the one metric implementation."""
    actual = np.asarray(target)
    base_rate = actual.mean()  # share of buyers in these rows
    metrics = {
        "pr_auc": float(average_precision_score(actual, scores)),
        "roc_auc": float(roc_auc_score(actual, scores)),
    }
    for share in cfg.contact_fractions:
        # Contacting this share of the customers: who is called, who of them buys, and how many
        # of all buyers that reaches. Precision and recall are both fixed by that one decision,
        # so F1 (their harmonic mean) summarises it in one number.
        contacted = top_share_mask(scores, share)
        buyers_contacted = actual[contacted].sum()
        precision = float(buyers_contacted / contacted.sum())
        recall = float(buyers_contacted / actual.sum())
        metrics[f"{share_key(share)}_precision"] = precision
        metrics[f"{share_key(share)}_recall"] = recall
        metrics[f"{share_key(share)}_f1"] = (
            float(2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        )
        metrics[f"{share_key(share)}_lift"] = float(precision / base_rate)
    return metrics


def make_folds(cfg: Config, seed: int) -> StratifiedGroupKFold:
    """Group-aware, stratified cross-validation folds with a fixed seed."""
    return StratifiedGroupKFold(n_splits=cfg.cv_folds, shuffle=True, random_state=seed)


def cross_validate_pipeline(
    pipeline: Pipeline,
    features: pd.DataFrame,
    target: pd.Series,
    groups: pd.Series,
    folds: StratifiedGroupKFold,
    cfg: Config,
) -> list[dict]:
    """Train a fresh copy of the pipeline on each fold and return one metrics dict per fold."""
    fold_metrics = []
    for train_rows, valid_rows in folds.split(features, target, groups):
        # clone() gives an untrained copy, so no fold can reuse what another fold learned.
        model = clone(pipeline).fit(features.iloc[train_rows], target.iloc[train_rows])
        scores = model.predict_proba(features.iloc[valid_rows])[:, 1]
        fold_metrics.append(ranking_metrics(target.iloc[valid_rows], scores, cfg))
    return fold_metrics


def summarise_folds(fold_metrics: list[dict], prefix: str) -> dict:
    """Mean and standard deviation of every fold metric, e.g. cv_pr_auc_mean and cv_pr_auc_std."""
    table = pd.DataFrame(fold_metrics)
    summary = {}
    for metric in table.columns:
        summary[f"{prefix}_{metric}_mean"] = float(table[metric].mean())
        summary[f"{prefix}_{metric}_std"] = float(table[metric].std(ddof=1))
    return summary


def model_params(pipeline: Pipeline) -> dict:
    """Hyperparameters of the model step, prefixed with 'model__' like in the search spaces."""
    # The prefix also keeps these names apart from the parameters MLflow autolog records for the
    # search itself (such as its own n_jobs), which MLflow would otherwise refuse as a conflict.
    params = pipeline.named_steps["model"].get_params()
    return {f"model__{name}": value for name, value in params.items()}


def setup_tracking(cfg: Config) -> None:
    """Point MLflow at its database in the work folder and select the experiment."""
    cfg.work_dir.mkdir(parents=True, exist_ok=True)  # SQLite does not create missing folders
    mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
    mlflow.set_experiment(cfg.mlflow_experiment)


def log_run(params: dict, metrics: dict) -> None:
    """Log parameters and metrics to the active MLflow run: the one place that writes them."""
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)


def tune_pipeline(
    pipeline: Pipeline,
    search_space: dict,
    features: pd.DataFrame,
    target: pd.Series,
    groups: pd.Series,
    folds: StratifiedGroupKFold,
    cfg: Config,
) -> Pipeline:
    """Search hyperparameters on the group-aware folds; return an untrained pipeline with the best."""
    search = RandomizedSearchCV(
        pipeline,
        search_space,
        n_iter=cfg.search_n_iter,
        scoring=cfg.scoring,
        cv=folds,
        refit=False,  # the best settings are re-evaluated below, no need to fit them here
        random_state=cfg.random_state,
    )
    # Autolog records the best tried combinations as child runs of the active run. It is switched
    # off right after the search, otherwise every later fit would create unwanted extra runs.
    mlflow.sklearn.autolog(
        max_tuning_runs=cfg.max_tuning_runs, log_models=False, log_datasets=False, silent=True
    )
    try:
        search.fit(features, target, groups=groups)  # groups reach the group-aware folds
    finally:
        mlflow.sklearn.autolog(disable=True)
    return clone(pipeline).set_params(**search.best_params_)


def evaluate_candidate(
    candidate: Candidate,
    features: pd.DataFrame,
    target: pd.Series,
    groups: pd.Series,
    folds: StratifiedGroupKFold,
    cfg: Config,
    data_revision: str,
) -> CandidateResult:
    """Tune if needed, cross-validate and log one candidate as its own MLflow run."""
    print(f"  {candidate.name:<24} ...", end="", flush=True)
    with mlflow.start_run(run_name=candidate.name) as run:
        mlflow.set_tags(
            {
                "feature_set": cfg.feature_set_name,
                "stage": candidate.stage,
                "model_family": candidate.family,
                "data_revision": data_revision,
            }
        )
        setup_params = {
            "n_features": len(cfg.feature_columns),
            "cv_folds": cfg.cv_folds,
            "test_size": cfg.test_size,
            "random_state": cfg.random_state,
            "scoring": cfg.scoring,
        }

        pipeline = candidate.pipeline
        if candidate.search_space:
            pipeline = tune_pipeline(pipeline, candidate.search_space, features, target, groups, folds, cfg)
            setup_params["search_n_iter"] = cfg.search_n_iter

        # Cross-validated performance: the basis for every selection decision.
        metrics = summarise_folds(
            cross_validate_pipeline(pipeline, features, target, groups, folds, cfg), "cv"
        )

        # In-sample performance, only to measure how much the model memorises.
        in_sample_scores = clone(pipeline).fit(features, target).predict_proba(features)[:, 1]
        in_sample = ranking_metrics(target, in_sample_scores, cfg)
        metrics["train_pr_auc"] = in_sample["pr_auc"]
        metrics["train_roc_auc"] = in_sample["roc_auc"]
        metrics["overfit_gap_pr_auc"] = metrics["train_pr_auc"] - metrics["cv_pr_auc_mean"]

        params = model_params(pipeline)
        log_run({**setup_params, **params}, metrics)

    print(f" cv PR-AUC {metrics['cv_pr_auc_mean']:.3f} (± {metrics['cv_pr_auc_std']:.3f})", flush=True)
    return CandidateResult(candidate, pipeline, params, metrics, run.info.run_id)


# =============================================================================
# Step 4 - Select
# =============================================================================
#
# Input : the five CandidateResults
# Output: the selected CandidateResult; every candidate's outcome is tagged in MLflow
#
# A. Gate. A candidate is rejected when it does not beat the dummy's CV PR-AUC by at least 0.10:
#    a model barely better than guessing the base rate is useless for choosing whom to call.
#    There is no gate on the fold-to-fold spread of PR-AUC. A trial run showed that with ~130
#    buyers per validation fold every good model varies by 0.06-0.09, so such a gate would reject
#    all of them; whether two models really differ is decided in the final (C) instead.
# B. Ranking. Candidates that pass are sorted by cv_pr_auc_mean; the best two are finalists.
# C. Final. Both finalists are cross-validated again on 15 folds (the 5-fold arrangement repeated
#    with 3 different seeds), always on the same folds, and their PR-AUC difference is computed
#    per fold.
#      - Clear winner: the average difference is larger than one standard error of the
#        differences -> the higher one wins.
#      - Tie: otherwise decide, in this order, on higher precision in the top 10%, lower CV
#        standard deviation, smaller overfit gap, and finally the simpler model family.
#    The standard error over overlapping folds is optimistic (folds share training rows), so it
#    is only used to decide whether the tie-break rules apply, never to claim that one model is
#    "significantly" better.
#
# Each candidate's outcome is written to MLflow as the tag selection_status (with
# rejected_reason where relevant), so the decision can be read back in the MLflow UI.
#

def apply_gates(results: list[CandidateResult], cfg: Config) -> list[CandidateResult]:
    """Mark the dummy as reference, reject candidates that fail the gate, return the rest."""
    dummy = next(result for result in results if result.candidate.family == "dummy")
    dummy.status = "reference"

    passed = []
    for result in results:
        if result is dummy:
            continue
        lift = result.metrics["cv_pr_auc_mean"] - dummy.metrics["cv_pr_auc_mean"]
        result.metrics["lift_over_dummy"] = lift
        if lift < cfg.min_lift_over_dummy:
            result.status, result.reason = "rejected_gate", f"lift_over_dummy={lift:.3f}"
        else:
            passed.append(result)
    return passed


def run_finale(
    first: CandidateResult,
    second: CandidateResult,
    features: pd.DataFrame,
    target: pd.Series,
    groups: pd.Series,
    cfg: Config,
) -> None:
    """Cross-validate both finalists on the same 15 folds and store the finale_* metrics."""
    per_fold = {first.candidate.name: [], second.candidate.name: []}
    for seed in cfg.finale_seeds:
        folds = make_folds(cfg, seed)  # same seed -> identical folds for both finalists
        for finalist in (first, second):
            per_fold[finalist.candidate.name] += cross_validate_pipeline(
                finalist.pipeline, features, target, groups, folds, cfg
            )

    tiebreak = f"{share_key(cfg.tiebreak_contact_fraction)}_precision"
    first_table = pd.DataFrame(per_fold[first.candidate.name])
    second_table = pd.DataFrame(per_fold[second.candidate.name])
    differences = first_table["pr_auc"] - second_table["pr_auc"]  # paired: same fold, two models

    for finalist, table in ((first, first_table), (second, second_table)):
        finalist.finale = {
            "finale_pr_auc_mean": float(table["pr_auc"].mean()),
            f"finale_{tiebreak}_mean": float(table[tiebreak].mean()),
        }
    first.finale["finale_diff_mean"] = float(differences.mean())
    first.finale["finale_diff_se"] = float(differences.std(ddof=1) / np.sqrt(len(differences)))


def break_tie(first: CandidateResult, second: CandidateResult, cfg: Config) -> tuple[CandidateResult, str]:
    """Pick the winner of the final and describe which rule decided."""
    diff, se = first.finale["finale_diff_mean"], first.finale["finale_diff_se"]
    if abs(diff) > cfg.tie_tolerance_se * se:
        winner = first if diff > 0 else second
        return winner, f"clear winner: PR-AUC difference {abs(diff):.4f} > {cfg.tie_tolerance_se:g} SE ({se:.4f})"

    tiebreak = f"finale_{share_key(cfg.tiebreak_contact_fraction)}_precision_mean"
    rules = [  # (description, value per finalist, higher is better?)
        (tiebreak, lambda r: r.finale[tiebreak], True),
        ("cv_pr_auc_std", lambda r: r.metrics["cv_pr_auc_std"], False),
        ("overfit_gap_pr_auc", lambda r: r.metrics["overfit_gap_pr_auc"], False),
        ("simplicity", lambda r: cfg.simplicity_order.index(r.candidate.family), False),
    ]
    for name, value, higher_is_better in rules:
        a, b = value(first), value(second)
        if a != b:
            winner = first if (a > b) == higher_is_better else second
            return winner, f"tie within {cfg.tie_tolerance_se:g} SE; decided by {name} ({a:.4f} vs {b:.4f})"
    return first, "tie on every rule; kept the higher-ranked finalist"


def tag_selection(results: list[CandidateResult]) -> None:
    """Write each candidate's selection outcome and the gate/finale metrics back to MLflow."""
    client = MlflowClient()
    for result in results:
        client.set_tag(result.run_id, "selection_status", result.status)
        if result.reason:
            client.set_tag(result.run_id, "rejected_reason", result.reason)
        extra_metrics = {**result.finale}
        if "lift_over_dummy" in result.metrics:
            extra_metrics["lift_over_dummy"] = result.metrics["lift_over_dummy"]
        for name, value in extra_metrics.items():
            client.log_metric(result.run_id, name, value)


def select_model(
    results: list[CandidateResult],
    features: pd.DataFrame,
    target: pd.Series,
    groups: pd.Series,
    cfg: Config,
) -> tuple[CandidateResult, str]:
    """Run gate, ranking and final; return the selected candidate and the deciding rule."""
    # A. Gate
    passed = apply_gates(results, cfg)
    if not passed:
        tag_selection(results)
        raise RuntimeError("No candidate passed the gates: no model learns more than the base rate")

    # B. Ranking
    ranked = sorted(passed, key=lambda result: result.metrics["cv_pr_auc_mean"], reverse=True)
    for position, result in enumerate(ranked[2:], start=3):
        result.status, result.reason = "ranked_out", f"rank {position} on cv_pr_auc_mean"

    # C. Final (only needed when there are two finalists)
    if len(ranked) == 1:
        selected, decision = ranked[0], "only candidate that passed the gates"
    else:
        first, second = ranked[0], ranked[1]
        print(f"  final: {first.candidate.name} vs {second.candidate.name} on "
              f"{len(cfg.finale_seeds) * cfg.cv_folds} folds ...", flush=True)
        run_finale(first, second, features, target, groups, cfg)
        first.status = second.status = "finalist"
        selected, decision = break_tie(first, second, cfg)

    selected.status = "selected"
    tag_selection(results)
    return selected, decision


# =============================================================================
# Step 5 - Operating points
# =============================================================================
#
# Input : the selected pipeline and the training data
# Output: a decision threshold per operating point, and a table with what each threshold means
#
# The model produces a score per customer; marketing needs a yes/no decision. A threshold turns
# scores into decisions, and which threshold is right depends on the situation:
#   max_f1      the threshold with the best balance between precision and recall (F1); a neutral
#               choice when the contact budget is unknown
#   top_5pct,   the threshold that selects the 5%, 10% or 20% highest-scoring customers; for a
#   top_10pct,  fixed contact budget (e.g. 10,000 calls), where the budget sets the threshold
#   top_20pct
# No single operating point is "the" answer: marketing picks one once the budget is known, and
# the model does not need retraining for that.
#
# Thresholds are set on out-of-fold scores: every training row is scored by a model that did not
# see it during training (cross_val_predict on the same group-aware folds). Scores of rows a model
# was trained on are too optimistic and would give thresholds that do not hold for new customers.
#
# Columns of the table:
#   threshold        score from which a customer is contacted
#   share_contacted  share of customers at or above the threshold
#   precision        share of contacted customers who buy
#   recall           share of all buyers that is contacted
#   lift             precision divided by the base rate: how many times better than calling at random
#   pct_of_ceiling   precision relative to the best possible precision for that share; when more
#                    customers are contacted than there are buyers, precision cannot reach 1
#

def out_of_fold_scores(
    pipeline: Pipeline,
    features: pd.DataFrame,
    target: pd.Series,
    groups: pd.Series,
    folds: StratifiedGroupKFold,
) -> np.ndarray:
    """Score every training row with a model that was not trained on that row."""
    probabilities = cross_val_predict(
        pipeline, features, target, groups=groups, cv=folds, method="predict_proba"
    )
    return probabilities[:, 1]


def threshold_for_max_f1(target: pd.Series, scores: np.ndarray) -> float:
    """Threshold with the highest F1 score (balance of precision and recall)."""
    precision, recall, thresholds = precision_recall_curve(target, scores)
    # The last precision/recall pair has no threshold; drop it so the arrays line up.
    f1 = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[np.argmax(f1)])


def threshold_for_top_share(scores: np.ndarray, share: float) -> float:
    """Score of the last customer inside the top `share` of the ranking."""
    n_contacted = max(1, round(share * len(scores)))
    return float(np.sort(scores)[::-1][n_contacted - 1])


def operating_thresholds(target: pd.Series, scores: np.ndarray, cfg: Config) -> dict:
    """All operating points and their thresholds, determined on out-of-fold scores."""
    thresholds = {"max_f1": threshold_for_max_f1(target, scores)}
    for share in cfg.contact_fractions:
        thresholds[share_key(share)] = threshold_for_top_share(scores, share)
    return thresholds


def decision_metrics(target: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    """What contacting every customer at or above the threshold would achieve."""
    actual = np.asarray(target)
    contacted = scores >= threshold
    base_rate = actual.mean()
    share_contacted = contacted.mean()
    precision = actual[contacted].mean() if contacted.any() else np.nan
    recall = actual[contacted].sum() / actual.sum()
    ceiling = min(1.0, base_rate / share_contacted) if share_contacted > 0 else np.nan
    return {
        "threshold": float(threshold),
        "share_contacted": float(share_contacted),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(2 * precision * recall / (precision + recall)) if precision + recall > 0 else 0.0,
        "lift": float(precision / base_rate),
        "pct_of_ceiling": float(precision / ceiling),
    }


def operating_point_table(target: pd.Series, scores: np.ndarray, thresholds: dict) -> pd.DataFrame:
    """One row per operating point with the metrics of decision_metrics."""
    rows = {name: decision_metrics(target, scores, threshold) for name, threshold in thresholds.items()}
    return pd.DataFrame(rows).T.rename_axis("operating_point")


# =============================================================================
# Step 6 - Test
# =============================================================================
#
# Input : the selected pipeline, the full training data, the thresholds from step 5, test.csv
# Output: test metrics, the operating-point table on test, uncertainty ranges and final checks
#
# The selected pipeline is trained once more on all training rows. test.csv is loaded here for the
# first time, and every threshold from step 5 is applied unchanged, exactly as it would be to new
# customers in production.
#
# Uncertainty ranges: the top 10% of the test set is only about 83 customers, so a single
# precision number would suggest more certainty than there is. The test set is resampled 1,000
# times with replacement; the 2.5th and 97.5th percentiles give a 95% range.
#
# Two fixed checks, each with a fixed response:
#   - CV was reliable: test PR-AUC within 0.05 of the CV estimate. Otherwise a warning is printed;
#     a different model is NOT chosen, because that would turn the test set into a validation set.
#   - No leakage: test ROC-AUC at most 0.90. Pre-contact customer profiles cannot plausibly predict
#     a purchase better than that; a higher value points to leaked information. The script then
#     stops before registering the model.
# The test result never changes which model was chosen; it only confirms the process or exposes
# an error in it.
#

def evaluate_on_test(
    model: Pipeline, test: pd.DataFrame, thresholds: dict, cfg: Config
) -> tuple[dict, pd.DataFrame, np.ndarray, pd.Series]:
    """Score the test set once; return test metrics, the operating-point table, scores and target."""
    features, target, _groups = split_columns(test, cfg)
    scores = model.predict_proba(features)[:, 1]

    table = operating_point_table(target, scores, thresholds)
    ranking = ranking_metrics(target, scores, cfg)
    predicted_buyer = scores >= thresholds["max_f1"]

    metrics = {
        "test_pr_auc": ranking["pr_auc"],
        "test_roc_auc": ranking["roc_auc"],
        "test_accuracy": float((predicted_buyer == np.asarray(target)).mean()),  # reference only
    }
    for point, row in table.iterrows():
        for column in ("precision", "recall", "lift", "pct_of_ceiling", "share_contacted"):
            metrics[f"test_{point}_{column}"] = float(row[column])
    return metrics, table, scores, target


def bootstrap_intervals(target: pd.Series, scores: np.ndarray, thresholds: dict, cfg: Config) -> dict:
    """95% ranges for precision and lift of each top-share operating point on the test set."""
    rng = np.random.default_rng(cfg.random_state)
    actual = np.asarray(target)
    intervals = {}
    for share in cfg.contact_fractions:
        point = share_key(share)
        precisions, lifts = [], []
        for _ in range(cfg.n_bootstrap):
            rows = rng.integers(0, len(actual), len(actual))  # resample with replacement
            resampled = decision_metrics(actual[rows], scores[rows], thresholds[point])
            precisions.append(resampled["precision"])
            lifts.append(resampled["lift"])
        for name, values in (("precision", precisions), ("lift", lifts)):
            low, high = np.nanpercentile(values, [2.5, 97.5])
            intervals[f"test_{point}_{name}_ci_low"] = float(low)
            intervals[f"test_{point}_{name}_ci_high"] = float(high)
    return intervals


def check_final(test_metrics: dict, cv_pr_auc_mean: float, cfg: Config) -> list[str]:
    """Run the two final checks: return warnings, or raise when leakage is suspected."""
    warnings = []
    gap = abs(test_metrics["test_pr_auc"] - cv_pr_auc_mean)
    test_metrics["cv_test_gap_pr_auc"] = gap
    if gap > cfg.max_cv_test_gap:
        warnings.append(
            f"test PR-AUC differs {gap:.3f} from the CV estimate (limit {cfg.max_cv_test_gap}); "
            "the CV estimate was less reliable than expected"
        )
    if test_metrics["test_roc_auc"] > cfg.leakage_roc_auc_ceiling:
        raise RuntimeError(
            f"test ROC-AUC {test_metrics['test_roc_auc']:.3f} exceeds {cfg.leakage_roc_auc_ceiling}: "
            "suspected data leakage. The model is NOT registered; check the features."
        )
    return warnings


def log_final_run(
    selected: CandidateResult,
    oof_table: pd.DataFrame,
    test_table: pd.DataFrame,
    test_metrics: dict,
    cfg: Config,
    data_revision: str,
) -> str:
    """Log the selected model's full evaluation as the run final_<name>; return its run id."""
    with mlflow.start_run(run_name=f"final_{selected.candidate.name}") as run:
        mlflow.set_tags(
            {
                "feature_set": cfg.feature_set_name,
                "stage": "final",
                "model_family": selected.candidate.family,
                "selection_status": "selected",
                "data_revision": data_revision,
            }
        )
        oof_metrics = {
            f"oof_{point}_{column}": float(value)
            for point, row in oof_table.iterrows()
            for column, value in row.items()
        }
        metrics = {**selected.metrics, **selected.finale, **oof_metrics, **test_metrics}
        log_run({"n_features": len(cfg.feature_columns), **selected.model_params}, metrics)

        # The operating-point table as a file: aggregated numbers per operating point, no rows
        # per customer.
        cfg.model_dir.mkdir(parents=True, exist_ok=True)
        table_path = cfg.model_dir / cfg.operating_points_file
        combined_operating_points(oof_table, test_table).to_csv(table_path)
        mlflow.log_artifact(str(table_path))
    return run.info.run_id


def combined_operating_points(oof_table: pd.DataFrame, test_table: pd.DataFrame) -> pd.DataFrame:
    """Out-of-fold and test operating points in one table, marked by a 'split' column."""
    return pd.concat(
        [oof_table.assign(split="train_out_of_fold"), test_table.assign(split="test")]
    ).reset_index().set_index(["split", "operating_point"])


# =============================================================================
# Step 7 - Register
# =============================================================================
#
# Input : the trained model, thresholds, test results and the MLflow run id
# Output: model.joblib, model_metadata.json and operating_points.csv in the private model
#         repository on the Hub; the commit id is the MODEL_REVISION
#
# Registering means publishing the chosen model to the place the app loads it from: a private
# model repository on the Hugging Face Hub. Like the dataset repository it keeps every upload as a
# commit, so the app can pin an exact model version (MODEL_REVISION) and a new training run never
# changes the app by surprise.
#
# model_metadata.json holds everything the app needs besides the model itself, so the app does
# not keep its own copy of feature lists or thresholds that could drift apart from the model:
#   - package and Python versions: a saved model only loads reliably with the same versions
#   - feature lists, the value range of every numeric feature and the category values the model
#     knows, so the app's input form only offers values the model has seen during training
#   - the operating-point thresholds and the base rate
#   - where the model came from: selected run, MLflow run id, data revision
#   - parity examples: a few made-up customer profiles with the score this model gives them, so
#     the deployed app can verify it reproduces exactly the same scores. They are synthetic
#     (medians and most common values), not real customers.
#

def input_bounds(features: pd.DataFrame, cfg: Config) -> dict:
    """Smallest and largest training value of each numeric feature, and whether it is a whole number."""
    # A tree model cannot extrapolate: for an age or income outside the range it was trained on it
    # simply repeats the prediction for the nearest value it knows. The app therefore limits every
    # numeric input field to the range seen in training. Missing values are ignored here.
    bounds = {}
    for column in cfg.numeric_features:
        values = features[column].dropna()
        bounds[column] = {
            "min": float(values.min()),
            "max": float(values.max()),
            "integer": bool((values % 1 == 0).all()),  # e.g. Age and CityTier take whole numbers only
        }
    return bounds


def category_levels(model: Pipeline, cfg: Config) -> dict:
    """Categories per categorical feature as learned by the fitted one-hot encoder."""
    encoder = model.named_steps["preprocess"].named_transformers_["categorical"].named_steps["encode"]
    return {
        feature: [str(level) for level in levels]
        for feature, levels in zip(cfg.categorical_features, encoder.categories_)
    }


def parity_examples(model: Pipeline, features: pd.DataFrame, cfg: Config) -> list[dict]:
    """A few synthetic customer profiles with the score the trained model gives them."""
    # A "typical" profile: median for numeric features, most common value for the others.
    typical = {column: float(features[column].median()) for column in cfg.numeric_features}
    for column in (*cfg.binary_features, *cfg.categorical_features):
        value = features[column].mode().iloc[0]
        typical[column] = int(value) if column in cfg.binary_features else str(value)

    # Two variations, so the check also covers a different binary and categorical value.
    with_passport = {**typical, "Passport": 1 - typical["Passport"]}
    levels = category_levels(model, cfg)["Designation"]
    other_designation = {**typical, "Designation": next(l for l in levels if l != typical["Designation"])}

    examples = []
    for profile in (typical, with_passport, other_designation):
        score = model.predict_proba(pd.DataFrame([profile])[cfg.feature_columns])[:, 1][0]
        examples.append({"input": profile, "expected_score": float(score)})
    return examples


def build_metadata(
    model: Pipeline,
    features: pd.DataFrame,
    target: pd.Series,
    thresholds: dict,
    selected: CandidateResult,
    final_run_id: str,
    data_revision: str,
    cfg: Config,
) -> dict:
    """Everything the app needs to use the model correctly, as a JSON-ready dict."""
    return {
        "python_version": ".".join(platform.python_version_tuple()[:2]),
        "packages": {
            package: version(package)
            for package in ("scikit-learn", "xgboost", "numpy", "pandas", "joblib")
        },
        "feature_columns": {
            "numeric": list(cfg.numeric_features),
            "binary": list(cfg.binary_features),
            "categorical": list(cfg.categorical_features),
        },
        "input_bounds": input_bounds(features, cfg),
        "category_levels": category_levels(model, cfg),
        "operating_points": thresholds,
        "prevalence": float(target.mean()),
        "selected_run": f"final_{selected.candidate.name}",
        "mlflow_run_id": final_run_id,
        "data_revision": data_revision,
        "parity_examples": parity_examples(model, features, cfg),
    }


def save_model_bundle(model: Pipeline, metadata: dict, cfg: Config) -> list:
    """Save model and metadata next to operating_points.csv in the model folder; return the paths."""
    cfg.model_dir.mkdir(parents=True, exist_ok=True)
    model_path = cfg.model_dir / cfg.model_file
    metadata_path = cfg.model_dir / cfg.metadata_file
    joblib.dump(model, model_path)
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    return [model_path, metadata_path, cfg.model_dir / cfg.operating_points_file]


def upload_model_bundle(api: HfApi, cfg: Config, paths: list) -> str:
    """Upload the bundle to the private model repository in one commit; return the commit id."""
    ensure_private_repo(api, cfg.hf_model_repo, repo_type="model")
    # The model file is ~20 MB; the Hub library would draw upload progress bars that fill the
    # notebook output and the workflow log with dozens of lines, so they are switched off.
    disable_progress_bars()
    # One commit for all files, so every model version on the Hub has matching metadata and
    # operating points.
    commit = api.create_commit(
        repo_id=cfg.hf_model_repo,
        repo_type="model",
        operations=[CommitOperationAdd(path_in_repo=path.name, path_or_fileobj=path) for path in paths],
        commit_message="Register selected model",
    )
    return commit.oid


# =============================================================================
# Reporting
# =============================================================================
#
# Everything printed here also goes to the GitHub Actions run summary (when running there), so the
# results can be read without opening MLflow. Only aggregated numbers are shown.
#

def runs_overview(results: list[CandidateResult], cfg: Config) -> pd.DataFrame:
    """One row per candidate: selection outcome and CV metrics next to its tuned hyperparameters."""
    searched = sorted({name for space in (cfg.rf_search_space, cfg.xgb_search_space) for name in space})
    rows = {}
    for result in results:
        row = {
            "status": result.status,
            "cv_pr_auc_mean": result.metrics["cv_pr_auc_mean"],
            "cv_pr_auc_std": result.metrics["cv_pr_auc_std"],
            "cv_roc_auc_mean": result.metrics["cv_roc_auc_mean"],
            # Precision, recall and F1 of one concrete decision: contacting the top 10% of the
            # ranking. They make the ranking metrics above tangible without fixing a threshold yet.
            **{
                f"cv_top10_{name}": result.metrics[
                    f"cv_{share_key(cfg.tiebreak_contact_fraction)}_{name}_mean"
                ]
                for name in ("precision", "recall", "f1")
            },
            "overfit_gap_pr_auc": result.metrics["overfit_gap_pr_auc"],
        }
        # Only hyperparameters that were searched are shown; a column stays empty for models
        # that do not have that hyperparameter.
        for name in searched:
            if name in result.model_params:
                row[name.removeprefix("model__")] = result.model_params[name]
        rows[result.candidate.name] = row
    table = pd.DataFrame(rows).T.rename_axis("run")
    return table.sort_values("cv_pr_auc_mean", ascending=False)


def report(title: str, body: str | pd.DataFrame) -> None:
    """Print a titled block and add the same block to the GitHub Actions summary."""
    text = body.to_string() if isinstance(body, pd.DataFrame) else body
    print(f"\n=== {title} ===\n{text}")
    markdown = markdown_table(body) if isinstance(body, pd.DataFrame) else f"```\n{body}\n```"
    write_step_summary(f"### {title}\n\n{markdown}")


def confusion_text(target: pd.Series, scores: np.ndarray, threshold: float) -> str:
    """Confusion matrix at a threshold, as readable text."""
    (tn, fp), (fn, tp) = confusion_matrix(target, scores >= threshold, labels=[0, 1])
    return (f"contacted & bought: {tp:>4}   contacted, did not buy: {fp:>4}\n"
            f"not contacted, would have bought: {fn:>4}   not contacted, did not buy: {tn:>4}")


# =============================================================================
# Run all steps
# =============================================================================
#
# main() is the only place where the steps are connected. The order is the protocol: test.csv is
# loaded only after the model and its thresholds have been fixed.
#

def main() -> None:
    """Run steps 1 to 7 in order."""
    cfg = Config()
    api = HfApi()  # picks up the token from HF_TOKEN or `hf auth login`
    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 30)
    setup_tracking(cfg)

    # 1. Load train
    features, target, groups, data_revision = load_training_data(api, cfg)
    print(f"Train data    : {len(features)} rows, {target.mean():.1%} buyers, "
          f"revision {data_revision[:8]}")
    folds = make_folds(cfg, cfg.random_state)  # shared by every candidate

    # 2. Build models
    ladder = model_ladder(target, cfg)

    # 3. Evaluate candidates
    print(f"\nEvaluating {len(ladder)} candidates on {cfg.cv_folds} group-aware folds "
          f"(tuned models try {cfg.search_n_iter} combinations first):")
    results = [
        evaluate_candidate(candidate, features, target, groups, folds, cfg, data_revision)
        for candidate in ladder
    ]

    # 4. Select
    selected, decision = select_model(results, features, target, groups, cfg)
    report("Candidates (hyperparameters next to CV metrics)", runs_overview(results, cfg))
    rejected = [f"{r.candidate.name}: {r.status} ({r.reason})" for r in results if r.reason]
    report("Selection", f"selected: {selected.candidate.name}\nrule    : {decision}"
           + ("\n" + "\n".join(rejected) if rejected else ""))

    # 5. Operating points on out-of-fold scores
    oof_scores = out_of_fold_scores(selected.pipeline, features, target, groups, folds)
    thresholds = operating_thresholds(target, oof_scores, cfg)
    oof_table = operating_point_table(target, oof_scores, thresholds)
    report("Operating points (train, out-of-fold)", oof_table)

    # 6. Test: train on all training rows, then load and score the test set once
    model = clone(selected.pipeline).fit(features, target)
    test = load_split(cfg, cfg.test_file, data_revision)
    test_metrics, test_table, test_scores, test_target = evaluate_on_test(model, test, thresholds, cfg)
    test_metrics.update(bootstrap_intervals(test_target, test_scores, thresholds, cfg))
    warnings = check_final(test_metrics, selected.metrics["cv_pr_auc_mean"], cfg)
    final_run_id = log_final_run(selected, oof_table, test_table, test_metrics, cfg, data_revision)

    ranges = pd.DataFrame(
        {
            share_key(share): {
                "precision": f"{test_metrics[f'test_{share_key(share)}_precision']:.3f} "
                             f"[{test_metrics[f'test_{share_key(share)}_precision_ci_low']:.3f}, "
                             f"{test_metrics[f'test_{share_key(share)}_precision_ci_high']:.3f}]",
                "lift": f"{test_metrics[f'test_{share_key(share)}_lift']:.2f} "
                        f"[{test_metrics[f'test_{share_key(share)}_lift_ci_low']:.2f}, "
                        f"{test_metrics[f'test_{share_key(share)}_lift_ci_high']:.2f}]",
            }
            for share in cfg.contact_fractions
        }
    ).T.rename_axis("operating_point")
    report("Test set (used once)",
           f"PR-AUC {test_metrics['test_pr_auc']:.3f} (CV estimate {selected.metrics['cv_pr_auc_mean']:.3f}) | "
           f"ROC-AUC {test_metrics['test_roc_auc']:.3f} | accuracy at max_f1 {test_metrics['test_accuracy']:.3f}\n"
           + ("warnings: " + "; ".join(warnings) if warnings else "checks: CV estimate reliable, no sign of leakage"))
    report("Operating points (test, thresholds from out-of-fold)", test_table)
    report("Test precision and lift with 95% ranges", ranges)
    report("Decisions at max_f1 (test)", confusion_text(test_target, test_scores, thresholds["max_f1"]))
    report("Decisions when contacting the top 10% (test)",
           confusion_text(test_target, test_scores, thresholds[share_key(cfg.tiebreak_contact_fraction)]))

    # 7. Register
    metadata = build_metadata(model, features, target, thresholds, selected, final_run_id, data_revision, cfg)
    paths = save_model_bundle(model, metadata, cfg)
    model_revision = upload_model_bundle(api, cfg, paths)
    report("Registered model",
           f"repository : https://huggingface.co/{cfg.hf_model_repo} (private)\n"
           f"files      : {', '.join(path.name for path in paths)}\n"
           f"MODEL_REVISION: {model_revision}\n"
           f"MLflow run : final_{selected.candidate.name} ({final_run_id})")

    # In GitHub Actions: tell the deployment job which model version to use.
    export_github_output("model_revision", model_revision)


if __name__ == "__main__":
    main()

Overwriting tourism_project/model_building/train.py


In [42]:
# === PHASE 05: RUN MODEL TRAINING ===
# Run the script written above: compare the candidates, select one, evaluate it once on the test set,
# log everything to MLflow and register the model on the Hub. %run stops the cell on an error.
%run -m tourism_project.model_building.train

Train data    : 3302 rows, 19.3% buyers, revision 0d0e7fe2

Evaluating 5 candidates on 5 group-aware folds (tuned models try 30 combinations first):
  dummy_baseline           ... cv PR-AUC 0.193 (± 0.001)
  randomforest_baseline    ... cv PR-AUC 0.612 (± 0.086)
  xgboost_baseline         ... cv PR-AUC 0.566 (± 0.058)
  randomforest_tuned       ... cv PR-AUC 0.615 (± 0.081)
  xgboost_tuned            ... cv PR-AUC 0.578 (± 0.067)
  final: randomforest_tuned vs randomforest_baseline on 15 folds ...

=== Candidates (hyperparameters next to CV metrics) ===
                           status cv_pr_auc_mean cv_pr_auc_std cv_roc_auc_mean cv_top10_precision cv_top10_recall cv_top10_f1 overfit_gap_pr_auc class_weight max_depth max_features min_samples_leaf min_samples_split n_estimators colsample_bytree gamma learning_rate min_child_weight reg_lambda subsample
run                                                                                                                                     

**Browsing the experiments (optional).** The tables above come from the same numbers MLflow stored.
To click through every run, hyperparameter and child run, start the MLflow web interface. In Colab the
next cell starts it in the background and opens it through Colab's proxy in a new browser tab;
locally it prints the command to run in a terminal.

In [43]:
# === PHASE 05b: MLFLOW UI (optional) ===
# Start the MLflow web interface on the tracking database that train.py wrote.
import subprocess
import time

from tourism_project.config import Config

tracking_uri = Config().mlflow_tracking_uri
if in_colab:
    # Run the UI as a background process so the notebook stays usable. --allowed-hosts is needed
    # because the Colab proxy forwards requests under its own host name.
    subprocess.Popen(
        ["mlflow", "ui", "--backend-store-uri", tracking_uri, "--port", "5000", "--allowed-hosts", "*"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(10)  # give the server a moment to start

    from google.colab import output

    output.serve_kernel_port_as_window(5000)
    print("MLflow UI started: use the link above to open it in a new tab.")
else:
    print(f"Run this in a terminal from the project root, then open http://127.0.0.1:5000:\n"
          f"  mlflow ui --backend-store-uri {tracking_uri}")

Run this in a terminal from the project root, then open http://127.0.0.1:5000:
  mlflow ui --backend-store-uri sqlite:////mnt/c/Users/richa/Mijn Drive/tourism/outputs/mlflow.db


# Deployment

## Dockerfile

**What the Dockerfile does.** A Hugging Face Space with `sdk: docker` builds a container image from this file
and runs the Streamlit app inside it. The image holds exactly what the app needs, and nothing else:

- **Python 3.12 and the package versions used in training** (see Dependency Handling). A saved model only scores
  identically with the versions it was trained with; the template's Python 3.9 cannot even install them.
- **Only `app.py` and its Streamlit settings are copied in.** `.dockerignore` whitelists those files, so customer
  data, the MLflow database or notebooks can never end up in this public image.
- **The model is not in the image.** The app downloads it at start-up from the private model repository, at the
  commit stored in the Space variable `MODEL_REVISION`. A newly trained model needs a new variable value, not a
  new image.
- **Settings for a public app.** It runs as a normal user (uid 1000, which Spaces require), keeps Streamlit's
  protection against cross-site requests switched on (the template switched it off), sends no usage statistics
  and shows visitors no error details.
- **Port 8501**, matched by `app_port: 8501` in the Space's `README.md`. Without that line Hugging Face sends
  traffic to port 7860 and the app never becomes reachable.

In [44]:
# === PHASE 06: DEPLOYMENT FOLDERS ===
# Create the folder for the deployment files, including the .streamlit subfolder for the app settings
# (%%writefile does not create missing folders).
os.makedirs("tourism_project/deployment/.streamlit", exist_ok=True)

print("Deployment folders ready: tourism_project/deployment, tourism_project/deployment/.streamlit")

Deployment folders ready: tourism_project/deployment, tourism_project/deployment/.streamlit


In [45]:
%%writefile tourism_project/deployment/Dockerfile
# Container image for the Wellness Tourism Streamlit app, built by the Hugging Face Space.
#
# The trained model is NOT part of this image. At start-up app.py downloads it from the private
# Hugging Face model repository at the exact commit set in the Space variable MODEL_REVISION.
# A new model therefore needs a new revision on the Hub and an updated variable, not a new image.

# Python version and package versions must match the training environment (model_metadata.json):
# a saved scikit-learn model only loads reliably with the versions it was saved with.
# "slim" is enough: the scikit-learn and XGBoost wheels bring their own compiled libraries.
FROM python:3.12-slim

# No .pyc files, unbuffered logs (visible immediately in the Space logs), smaller image, quieter pip.
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1 \
    PIP_DISABLE_PIP_VERSION_CHECK=1

# Dependencies first: Docker caches this layer and only reinstalls when requirements.txt changes,
# not on every change to app.py. They are installed as root, so the app user cannot alter them.
COPY requirements.txt /tmp/requirements.txt
RUN pip install -r /tmp/requirements.txt && rm /tmp/requirements.txt

# Hugging Face Spaces run containers as user id 1000; running as a normal user instead of root
# limits what a compromised app could do.
RUN useradd --create-home --uid 1000 user
USER user
ENV HOME=/home/user \
    HF_HOME=/home/user/.cache/huggingface
WORKDIR /home/user/app

# Only the two files the app needs. Together with the whitelist in .dockerignore this guarantees that
# nothing else from the project (customer data, experiment database, notebooks) ends up in the image.
COPY --chown=user app.py .
COPY --chown=user .streamlit/config.toml .streamlit/config.toml

# Streamlit listens on 8501; the Space README sets app_port: 8501 so Hugging Face routes traffic here.
EXPOSE 8501

# Start the app on all network interfaces (required inside a container). Security settings such as
# XSRF protection and hidden error details are in .streamlit/config.toml.
CMD ["streamlit", "run", "app.py", \
     "--server.port=8501", \
     "--server.address=0.0.0.0", \
     "--server.headless=true", \
     "--browser.gatherUsageStats=false"]

Overwriting tourism_project/deployment/Dockerfile


The next three cells write the files the image and the Space depend on: the whitelist of files that may enter
the image (`.dockerignore`), the Streamlit settings for a public app (`.streamlit/config.toml`), and the Space's
`README.md`, whose front matter (`sdk: docker`, `app_port: 8501`) configures how Hugging Face builds and serves it.

In [46]:
%%writefile tourism_project/deployment/.dockerignore
# Whitelist: ignore everything, then allow exactly the files the image is built from.
# The app is public, and the project also contains customer data, an experiment database and
# notebooks. With a whitelist none of that can end up in the image, even when files are added later.
*
!app.py
!requirements.txt
!.streamlit/config.toml

Overwriting tourism_project/deployment/.dockerignore


In [47]:
%%writefile tourism_project/deployment/.streamlit/config.toml
# Streamlit settings for the public app.

[server]
# No browser window to open inside a container.
headless = true
# Protection against cross-site request forgery stays on for a public app.
enableXsrfProtection = true

[browser]
# Do not send usage statistics to Streamlit.
gatherUsageStats = false

[client]
# Visitors see a generic message on an error; details (which could reveal internals) only go to the
# server log. Note: `false` would mean "show the stack trace" in this Streamlit version.
showErrorDetails = "none"

Overwriting tourism_project/deployment/.streamlit/config.toml


In [48]:
%%writefile tourism_project/deployment/README.md
---
title: Score a Lead
emoji: 🧳
colorFrom: green
colorTo: blue
sdk: docker
app_port: 8501
pinned: false
short_description: Which Wellness Tourism leads to contact first
---

# Score a Lead

Streamlit app for the marketing team of *Visit with Us*. A lead is a customer who has not been
contacted yet. Enter the profile of one lead and see whether that lead belongs to the group most
likely to buy the Wellness Tourism Package, so the sales team can decide whom to contact first.

- The prediction uses only information known **before** a customer is contacted.
- The model is a random forest selected and evaluated in the project's MLOps pipeline; it is loaded
  from a private Hugging Face model repository at a fixed version (`MODEL_REVISION`).
- The app shows a ranking score and the contact group the lead falls into, together with how
  well that group performed on held-out test customers. The score is not a probability.

The front matter above configures this Space: `sdk: docker` builds the included `Dockerfile`, and
`app_port: 8501` routes traffic to Streamlit's port.

Overwriting tourism_project/deployment/README.md


## Streamlit App

Please ensure that the web app script is named `app.py`.

**What the app does.** The app is called *Score a Lead*: a lead is a customer who has not been contacted
yet. Marketing enters the profile of one lead and sees whether that lead belongs to the group most likely to
buy the Wellness Tourism Package, and is therefore worth contacting first.

- **Input.** A form with the 14 pre-contact fields. Dropdowns offer exactly the categories the model knows, and
  each numeric field shows the range seen in training (for example monthly income 16,009 – 38,291); both come
  from `model_metadata.json`, not from the app code. The answers become a one-row DataFrame in the model's
  column order, which is what the model pipeline expects.
- **Values outside the training data get no score.** If a numeric value lies outside the range seen in training,
  the app names the field and the range instead of scoring: the model has no information about such customers,
  so a score would be a guess presented as an assessment. (A hard limit on the input field is avoided on purpose,
  because Streamlit would silently replace an out-of-range value by the previous one and score that instead.)
- **Output.** A ranking score and the most selective contact group the lead falls into (top 5%, 10% or 20%
  of customers), with how that group did on the test set: the share of contacted customers who bought, and how
  many times better than calling at random that is. The score is deliberately not shown as a probability: the
  model ranks customers well, but its scores were not calibrated to real purchase rates.
- **Checks at start-up.** The app compares the installed package versions with those used in training and
  re-scores the synthetic check profiles stored with the model. Any difference stops the app with an
  explanation: an app that looks fine but scores differently from the trained model is worse than no app.
- **Deliberately absent.** File upload (a public page should not invite people to send customer files) and any
  logging of what users enter.

In [49]:
%%writefile tourism_project/deployment/app.py
"""Streamlit app "Score a Lead": should the sales team contact this lead first?

Context
    Visit with Us is introducing the Wellness Tourism Package. Contacting every customer is expensive,
    so marketing wants to know, BEFORE a customer is contacted, whether that customer belongs to the
    group most likely to buy. This app is the last stage of the MLOps pipeline:

        data_register.py  ->  prep.py  ->  train.py         ->  this app
                                           model on the Hub      (Hugging Face Space)

    A lead is a customer who has not been contacted yet. A user enters the profile of one lead; the app
    returns a ranking score and the contact group the lead falls into (top 5%, 10% or 20% of customers),
    together with how well that group did on test customers the model never saw during training.

Where things come from
    - The trained model, its metadata and its operating points are downloaded from the private
      Hugging Face model repository at the commit given in the MODEL_REVISION environment variable
      (a Space variable). Pinning the commit means a new training run never changes the app silently.
    - model_metadata.json holds everything the app needs to know about the model: feature lists,
      allowed input ranges, category values, thresholds and package versions. The app keeps no copy
      of these, so it can never drift apart from the model.
    - HF_TOKEN (a Space secret with read access to the model repository only) is read from the
      environment and never shown or logged.

Why the score is not called a probability
    The model ranks customers well, but its scores were not calibrated to match real purchase rates.
    The app therefore explains a score through the contact groups and their measured precision, not
    as "x% chance to buy".

Inputs outside the training data
    Every numeric input must lie within the range seen in training (e.g. monthly income 16,009 - 38,291).
    Otherwise the app gives no score and says which value is outside which range: the model has no
    information about such customers, so a score would be a guess presented as an assessment.

Deliberately left out
    No file upload: a public app that accepts customer files invites people to send customer data to
    a public server. Scoring a whole customer list belongs in an internal job, not in this page.
    Inputs are never printed or logged.
"""
import json
import os
from dataclasses import dataclass
from importlib.metadata import version
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download


# =============================================================================
# Settings
# =============================================================================
#
# The container only contains this file, so the project's central Config is not available. The few
# fixed values below describe the model repository layout that train.py writes; everything about the
# model itself comes from model_metadata.json.
#

DEFAULT_MODEL_REPO = "richvrb/tourism-wellness-model"
MODEL_FILE = "model.joblib"
METADATA_FILE = "model_metadata.json"
OPERATING_POINTS_FILE = "operating_points.csv"

# A score recomputed here may differ from the training environment only by rounding noise.
PARITY_TOLERANCE = 1e-9

# How the operating points are shown to marketing, from most to least selective.
CONTACT_GROUPS = {
    "top_5pct": "Top 5% of customers",
    "top_10pct": "Top 10% of customers",
    "top_20pct": "Top 20% of customers",
}
BALANCED_POINT = ("max_f1", "Best balance of precision and recall")

# Readable labels and the section of the form each feature appears in. Features are grouped the way a
# sales employee thinks about a customer. Any feature not listed here still gets a field, under "Other".
FORM_SECTIONS = {
    "Customer profile": {
        "Age": "Age",
        "Gender": "Gender",
        "MaritalStatus": "Marital status",
        "Occupation": "Occupation",
        "Designation": "Job title",
        "MonthlyIncome": "Gross monthly income",
    },
    "Travel habits": {
        "NumberOfTrips": "Trips per year",
        "PreferredPropertyStar": "Preferred hotel rating (stars)",
        "Passport": "Holds a valid passport",
        "OwnCar": "Owns a car",
    },
    "Planned trip": {
        "NumberOfPersonVisiting": "People travelling",
        "NumberOfChildrenVisiting": "Children under 5 travelling",
    },
    "Contact and location": {
        "TypeofContact": "How the customer got in touch",
        "CityTier": "City tier (1 = most developed)",
    },
}


@dataclass(frozen=True)
class AppSettings:
    """Values the Space provides through environment variables."""

    model_repo: str
    model_revision: str | None
    token: str | None


def read_settings() -> AppSettings:
    """Read the model repository, its pinned revision and the access token from the environment."""
    return AppSettings(
        model_repo=os.environ.get("MODEL_REPO", DEFAULT_MODEL_REPO),
        model_revision=os.environ.get("MODEL_REVISION"),
        token=os.environ.get("HF_TOKEN"),  # None locally: then the `hf auth login` token is used
    )


# =============================================================================
# Loading the model
# =============================================================================
#
# st.cache_resource keeps the loaded model in memory for the lifetime of the container. Without it,
# Streamlit would download and load the ~20 MB model again on every click, because it reruns the whole
# script on each interaction.
#

@st.cache_resource(show_spinner="Loading the model from the Hugging Face Hub ...")
def load_bundle(model_repo: str, model_revision: str, _token: str | None) -> tuple:
    """Download model, metadata and operating points at the pinned revision; load them once."""
    # The leading underscore in _token tells Streamlit not to use the token as part of the cache key.
    def download(filename: str) -> str:
        return hf_hub_download(model_repo, filename, revision=model_revision, token=_token)

    model = joblib.load(download(MODEL_FILE))
    metadata = json.loads(Path(download(METADATA_FILE)).read_text(encoding="utf-8"))
    operating_points = pd.read_csv(download(OPERATING_POINTS_FILE))
    return model, metadata, operating_points


def feature_order(metadata: dict) -> list[str]:
    """Model input columns in the order the model was trained with."""
    groups = metadata["feature_columns"]
    return [*groups["numeric"], *groups["binary"], *groups["categorical"]]


# =============================================================================
# Start-up checks
# =============================================================================
#
# A model that loads without an error can still give different scores when the installed package
# versions differ from the training environment. Two checks run before the app shows anything:
#   1. the installed versions of the packages the model depends on must equal those in the metadata;
#   2. the synthetic example profiles stored at training time must get exactly the same score here.
# If either fails the app stops with an explanation: no app is better than an app with wrong scores.
#

def version_mismatches(metadata: dict) -> list[str]:
    """Packages whose installed version differs from the version the model was trained with."""
    return [
        f"{package}: installed {version(package)}, model trained with {trained_version}"
        for package, trained_version in metadata["packages"].items()
        if version(package) != trained_version
    ]


def parity_mismatches(model: object, metadata: dict) -> list[str]:
    """Example profiles whose score here differs from the score at training time."""
    mismatches = []
    for number, example in enumerate(metadata["parity_examples"], start=1):
        row = pd.DataFrame([example["input"]])[feature_order(metadata)]
        score = float(model.predict_proba(row)[:, 1][0])
        if abs(score - example["expected_score"]) > PARITY_TOLERANCE:
            mismatches.append(f"example {number}: expected {example['expected_score']:.6f}, got {score:.6f}")
    return mismatches


# =============================================================================
# Input form
# =============================================================================
#
# The form only offers what the model knows, and checks what it cannot restrict:
#   - categorical fields: exactly the categories the model's encoder learned (category_levels);
#   - yes/no fields: a toggle that becomes 1 or 0;
#   - numeric fields: accept any number, and show the range seen in training (input_bounds) as help.
#
# Numeric fields deliberately have no hard minimum or maximum. Streamlit silently replaces a value
# outside such limits by the previous valid value, so a customer with an income of 40,000 would be
# scored as if the income were the default 22,369, without any notice. Instead, the values are checked
# after submit (out_of_range_fields): if any value lies outside the range in the training data, the app
# shows which field and range, and gives NO score. The model has never seen such customers, so any score
# for them would be a guess presented as an assessment. The same check also catches typing mistakes
# such as "30.000" being read as 30.
#
# The form starts with a "typical" customer (median and most common values), the first parity example.
# The answers are returned as a one-row DataFrame in the model's column order, which is the format the
# model pipeline expects.
#

def section_of_features(metadata: dict) -> dict[str, dict[str, str]]:
    """Form sections with (feature -> label); features not placed in a section go to 'Other'."""
    placed = {feature for fields in FORM_SECTIONS.values() for feature in fields}
    sections = {
        title: {feature: label for feature, label in fields.items() if feature in feature_order(metadata)}
        for title, fields in FORM_SECTIONS.items()
    }
    unplaced = [feature for feature in feature_order(metadata) if feature not in placed]
    if unplaced:
        sections["Other"] = {feature: feature for feature in unplaced}
    return sections


def input_step(bounds: dict) -> float:
    """Step for the +/- buttons, scaled to the range: 1 for age, 100 for monthly income."""
    spread = bounds["max"] - bounds["min"]
    magnitude = 10 ** (len(str(int(spread))) - 3)  # a step of roughly 1/100 to 1/1000 of the range
    return max(1, magnitude) if bounds["integer"] else max(0.01, magnitude)


def feature_label(feature: str) -> str:
    """Readable label of a feature, as shown in the form."""
    for fields in FORM_SECTIONS.values():
        if feature in fields:
            return fields[feature]
    return feature


def out_of_range_fields(customer: pd.DataFrame, metadata: dict) -> list[str]:
    """Messages for numeric inputs outside the range seen in training; empty when all are inside."""
    messages = []
    for feature, bounds in metadata["input_bounds"].items():
        value = float(customer[feature].iloc[0])
        if not bounds["min"] <= value <= bounds["max"]:
            messages.append(
                f"{feature_label(feature)} {value:,.0f} is outside the range in the training data "
                f"({bounds['min']:,.0f} – {bounds['max']:,.0f})."
            )
    return messages


def input_widget(feature: str, label: str, metadata: dict, default: object) -> object:
    """One form field whose type and allowed values follow the model metadata."""
    groups = metadata["feature_columns"]

    if feature in groups["numeric"]:
        bounds = metadata["input_bounds"][feature]
        help_text = f"Range in the training data: {bounds['min']:,.0f} – {bounds['max']:,.0f}"
        step = input_step(bounds)
        # No min_value/max_value on purpose: see the section comment above.
        if bounds["integer"]:
            return st.number_input(label, value=int(round(default)), step=int(step), help=help_text)
        return st.number_input(label, value=float(default), step=step, help=help_text)

    if feature in groups["binary"]:
        return int(st.toggle(label, value=bool(default)))

    options = metadata["category_levels"][feature]
    index = options.index(default) if default in options else 0
    return st.selectbox(label, options, index=index)


def customer_form(metadata: dict) -> pd.DataFrame | None:
    """Show the input form; return the lead as a one-row DataFrame after submit, else None."""
    typical_customer = metadata["parity_examples"][0]["input"]
    answers = {}
    with st.form("customer"):
        for title, fields in section_of_features(metadata).items():
            st.subheader(title)
            columns = st.columns(2)
            for position, (feature, label) in enumerate(fields.items()):
                with columns[position % 2]:
                    answers[feature] = input_widget(feature, label, metadata, typical_customer[feature])
        submitted = st.form_submit_button("Score this lead", type="primary")

    if not submitted:
        return None
    return pd.DataFrame([answers])[feature_order(metadata)]


# =============================================================================
# Score and explanation
# =============================================================================
#
# The model gives a score between 0 and 1; a higher score means the customer resembles past buyers
# more. A score becomes a decision through the thresholds fixed during training:
#   - top_5pct / top_10pct / top_20pct: the score from which a customer belongs to the 5%, 10% or 20%
#     highest-scoring customers, for when marketing has a fixed calling budget;
#   - max_f1: the threshold with the best balance between finding buyers and avoiding wasted calls.
# For each group the app shows how it did on the test set: precision (share of contacted customers
# who bought) and lift (how many times better than contacting customers at random).
#

def score_customer(model: object, customer: pd.DataFrame) -> float:
    """Ranking score of one customer: higher means more similar to customers who bought."""
    return float(model.predict_proba(customer)[:, 1][0])


def test_results(operating_points: pd.DataFrame) -> pd.DataFrame:
    """Test-set precision and lift per operating point, indexed by operating point."""
    return operating_points[operating_points["split"] == "test"].set_index("operating_point")


def operating_point_view(score: float, metadata: dict, operating_points: pd.DataFrame) -> pd.DataFrame:
    """Table: for each contact group, its threshold, whether this lead is in it, and test results."""
    thresholds = metadata["operating_points"]
    results = test_results(operating_points)
    rows = []
    for key, label in [*CONTACT_GROUPS.items(), BALANCED_POINT]:
        rows.append(
            {
                "Contact group": label,
                "Score needed": round(thresholds[key], 3),
                "This lead": "yes" if score >= thresholds[key] else "no",
                "Buyers among contacted (test)": f"{results.loc[key, 'precision']:.0%}",
                "Better than random (test)": f"{results.loc[key, 'lift']:.1f}x",
            }
        )
    return pd.DataFrame(rows)


def show_verdict(score: float, metadata: dict, operating_points: pd.DataFrame) -> None:
    """Headline message: the most selective contact group this lead belongs to."""
    thresholds = metadata["operating_points"]
    results = test_results(operating_points)

    for key, label in CONTACT_GROUPS.items():  # most selective group first
        if score >= thresholds[key]:
            st.success(
                f"**{label}** — contact this lead. In the test set, "
                f"{results.loc[key, 'precision']:.0%} of the customers in this group bought the package, "
                f"{results.loc[key, 'lift']:.1f} times the average rate."
            )
            return

    if score >= thresholds[BALANCED_POINT[0]]:
        st.info(
            "**Outside the top 20%, but above the balanced threshold** — worth contacting when the "
            "calling budget is larger than 20% of customers."
        )
    else:
        st.warning("**Low priority** — outside the top 20% and below the balanced threshold.")


# =============================================================================
# Page
# =============================================================================

def model_info(settings: AppSettings, metadata: dict) -> None:
    """Collapsible section describing which model answers, so results can be traced."""
    with st.expander("About the model"):
        st.markdown(
            f"- Model repository: `{settings.model_repo}` at revision `{settings.model_revision[:8]}`\n"
            f"- Selected in training run: `{metadata['selected_run']}`\n"
            f"- Trained on dataset revision `{metadata['data_revision'][:8]}`; "
            f"{metadata['prevalence']:.0%} of training customers bought the package\n"
            f"- Uses only information known before contact: "
            f"{len(feature_order(metadata))} customer features, no sales-pitch details\n"
            f"- Python {metadata['python_version']}, "
            + ", ".join(f"{package} {v}" for package, v in metadata["packages"].items())
        )


def main() -> None:
    """Load and check the model, show the form, and explain the score of a submitted lead."""
    st.set_page_config(page_title="Score a Lead – Wellness Tourism", page_icon="🧳")
    st.title("Score a Lead")
    st.write(
        "A lead is a customer who has not been contacted yet. Enter the profile of one lead to see whether "
        "they belong to the group most likely to buy the Wellness Tourism Package, so the sales team knows "
        "whom to contact first. Only information known before the first contact is used."
    )

    settings = read_settings()
    if not settings.model_revision:
        st.error(
            "No model version configured. Set the Space variable MODEL_REVISION to a commit of the "
            "model repository (the hosting script does this)."
        )
        st.stop()

    try:
        model, metadata, operating_points = load_bundle(
            settings.model_repo, settings.model_revision, settings.token
        )
    except Exception:  # shown generically on purpose: the details stay in the server log
        st.error(
            "The model could not be loaded. Check that the Space secret HF_TOKEN can read the model "
            "repository and that MODEL_REVISION is an existing commit."
        )
        raise

    problems = version_mismatches(metadata) + parity_mismatches(model, metadata)
    if problems:
        st.error("The app stopped because it would not reproduce the trained model's scores:\n\n- "
                 + "\n- ".join(problems))
        st.stop()

    customer = customer_form(metadata)
    if customer is not None:
        outside = out_of_range_fields(customer, metadata)
        if outside:
            st.error(
                "**No score for this lead.** The model was trained on customers within these ranges "
                "and cannot reliably assess values outside them:\n\n- " + "\n- ".join(outside)
            )
            model_info(settings, metadata)
            st.stop()

        score = score_customer(model, customer)
        st.header("Result")
        show_verdict(score, metadata, operating_points)
        st.metric("Ranking score", f"{score:.3f}", help="Between 0 and 1. Higher means more similar to "
                  "customers who bought. It is a ranking, not a probability of buying.")
        st.dataframe(operating_point_view(score, metadata, operating_points), hide_index=True)

    model_info(settings, metadata)


if __name__ == "__main__":
    main()

Overwriting tourism_project/deployment/app.py


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

**Why exact versions.** The first five packages are pinned to the versions the model was trained with, which
are recorded in `model_metadata.json`. scikit-learn and XGBoost models are saved in Python's pickle format, and a
pickled model only loads reliably, and scores identically, with the same package versions; the app checks this
when it starts. Streamlit and huggingface_hub are what the app itself needs. MLflow is left out on purpose: the
app does not track experiments, and it would add hundreds of megabytes to the image.

In [50]:
%%writefile tourism_project/deployment/requirements.txt
# Packages for the Streamlit app, pinned to exact versions.
#
# The first five must be identical to the training environment ("packages" in model_metadata.json):
# a saved model only loads reliably, and scores identically, with the versions it was saved with.
# app.py checks this at start-up and refuses to run on a mismatch.
scikit-learn==1.9.1
xgboost==3.4.1
numpy==2.5.1
pandas==3.0.3
joblib==1.6.0

# App only. MLflow is deliberately absent: the app does not track experiments.
streamlit==1.63.0
huggingface_hub==1.31.0

Overwriting tourism_project/deployment/requirements.txt


# Hosting

**What hosting does.** The script below turns the deployment files into a running public app:

1. makes sure the Docker Space `richvrb/tourism-wellness-app` exists and is public;
2. picks the model version: the newest commit of the model repository (in GitHub Actions: the commit the
   training job has just registered);
3. stores that commit as the Space variable `MODEL_REVISION`, and checks that the Space secret `HF_TOKEN` exists;
4. uploads only the whitelisted deployment files, in one commit, which starts a new build of the image;
5. follows the build until the app runs, then checks that the public address answers without any login.

**Two tokens, on purpose.** Uploading needs write access, which comes from your own token. The running app uses
a different token, the Space secret `HF_TOKEN`, which can only *read* the model repository. The script never sets
that secret, so the public app never holds a token that could change anything.

Building the image and starting the app takes a few minutes. The output shows the model version, each change in
the Space status, the health check and the public address of the app.

In [51]:
# === PHASE 07a: HOSTING FOLDER ===
# Create the folder for the hosting script written in the next cell.
os.makedirs("tourism_project/hosting", exist_ok=True)

print("Hosting folder ready: tourism_project/hosting")

Hosting folder ready: tourism_project/hosting


In [52]:
%%writefile tourism_project/hosting/hosting.py
"""Hosting stage of the Visit with Us MLOps pipeline: publish the Streamlit app on Hugging Face.

Context
    The pipeline ends with an app that marketing can use: enter a customer profile, see whether that
    customer is worth contacting first. The app runs in a public Hugging Face Space, which builds a
    Docker image from the files in tourism_project/deployment/ and starts it.

        data_register.py  ->  prep.py  ->  train.py         ->  hosting.py
                                           model on the Hub      app in a public Space

    This script pushes the deployment files to that Space and tells the app which model version to
    load. It runs from the notebook (%run -m) and later as the last job in GitHub Actions (python -m).

What it does
    1. Space          - make sure the Docker Space exists and is public
    2. Model version  - pick the model commit the app must load (MODEL_REVISION)
    3. Space settings - store that commit as a Space variable; check the read-only token secret exists
    4. Upload         - upload only the whitelisted deployment files, in one commit
    5. Wait           - follow the build until the app runs, then check the public address answers

Tokens
    Uploading needs a token with write access (HF_TOKEN or `hf auth login`). The app itself uses a
    different token: the Space secret HF_TOKEN, which may only READ the model repository. This script
    checks that the secret exists but never sets it: the token available here (e.g. in GitHub Actions)
    has write access, and a public app must not hold a write token.

How the file is organised
    Settings come from the central Config (section "Deployment and hosting"). Each step has a header
    with its input, output and reasoning; main() connects the steps.

Running it
    python -m tourism_project.hosting.hosting   (from the project root)
    Building the image and starting the app takes a few minutes; the script waits and reports.
"""
import time
import urllib.error
import urllib.request

from huggingface_hub import HfApi
from huggingface_hub.utils import disable_progress_bars

from tourism_project.config import Config
from tourism_project.hub import resolve_model_revision

# Space states that mean the build or the app failed; waiting longer will not help.
FAILED_STAGES = {"BUILD_ERROR", "RUNTIME_ERROR", "CONFIG_ERROR", "NO_APP_FILE", "DELETING", "STOPPED", "PAUSED"}
# Space states that show a new build or restart is in progress.
BUSY_STAGES = {"BUILDING", "APP_STARTING", "RUNNING_BUILDING", "RUNNING_APP_STARTING"}
# If no build or restart becomes visible within this time, the current state is taken as final.
RESTART_GRACE_S = 180


# =============================================================================
# Step 1 - Space
# =============================================================================
#
# Input : the Space name from Config
# Output: a public Docker Space (created when missing)
#
# Creating a Docker Space requires a Hugging Face PRO account. Visibility is set on every run, so the
# app stays reachable even if the setting was changed by hand on the website.
#

def ensure_public_space(api: HfApi, cfg: Config) -> None:
    """Create the Docker Space if it does not exist, and make sure it is public."""
    api.create_repo(cfg.hf_space_repo, repo_type="space", space_sdk="docker", exist_ok=True)
    api.update_repo_settings(cfg.hf_space_repo, repo_type="space", visibility="public")


# =============================================================================
# Step 2 - Model version
# =============================================================================
#
# Input : the model repository, and $MODEL_REVISION when the training job passed one on
# Output: the commit id of the model the app has to load
#
# In GitHub Actions the training job passes on the commit it just registered, so the app serves exactly
# that model. From the notebook the newest commit of the model repository is used.
#

# resolve_model_revision lives in hub.py, next to the matching helper for the dataset revision.


# =============================================================================
# Step 3 - Space settings
# =============================================================================
#
# Input : the model commit
# Output: the Space variables MODEL_REVISION and MODEL_REPO set; the secret HF_TOKEN confirmed to exist
#
# Variables are visible settings (the app reads them as environment variables); secrets are hidden
# ones. Changing a variable makes the Space restart, so the app picks up the new model version.
#

def configure_space(api: HfApi, cfg: Config, model_revision: str) -> bool:
    """Set the model variables and check the token secret exists; return True if a variable changed."""
    wanted = {"MODEL_REPO": cfg.hf_model_repo, "MODEL_REVISION": model_revision}
    current = {key: variable.value for key, variable in api.get_space_variables(cfg.hf_space_repo).items()}

    changed = False
    for key, value in wanted.items():
        if current.get(key) != value:
            api.add_space_variable(cfg.hf_space_repo, key, value)
            changed = True

    # Only the NAME of the secret can be read back; its value stays hidden, which is the point.
    secret_names = set(api.get_space_secrets(cfg.hf_space_repo))
    if "HF_TOKEN" not in secret_names:
        raise RuntimeError(
            "The Space has no secret HF_TOKEN. Add a fine-grained token with READ access to "
            f"{cfg.hf_model_repo} under Settings -> Variables and secrets of {cfg.hf_space_repo}."
        )
    return changed


# =============================================================================
# Step 4 - Upload
# =============================================================================
#
# Input : the files listed in Config.deployment_files, from tourism_project/deployment/
# Output: those files in the Space repository, in one commit (which starts a new build)
#
# Only whitelisted files are uploaded, never the whole project: the project also holds customer data
# and an experiment database, and the Space is public. If no file changed, the Hub skips the commit.
#

def upload_deployment_files(api: HfApi, cfg: Config) -> tuple[str, bool]:
    """Upload the whitelisted deployment files; return the Space commit and whether it is new."""
    missing = [name for name in cfg.deployment_files if not (cfg.deployment_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Deployment files missing in {cfg.deployment_dir}: {missing}")

    commit_before = api.space_info(cfg.hf_space_repo).sha
    disable_progress_bars()  # keep notebook output and workflow logs readable
    commit = api.upload_folder(
        repo_id=cfg.hf_space_repo,
        repo_type="space",
        folder_path=cfg.deployment_dir,
        allow_patterns=list(cfg.deployment_files),  # whitelist: nothing else can be published
        commit_message="Deploy Streamlit app",
    )
    return commit.oid, commit.oid != commit_before


# =============================================================================
# Step 5 - Wait for the app
# =============================================================================
#
# Input : whether anything changed in steps 3 and 4
# Output: the app running, and its public address answering the health check
#
# After a change the Space first rebuilds or restarts. Until that starts, the status still shows the
# OLD state: the old app running, or "no app file" before the very first deployment. The script therefore
# only trusts "running" or an error status after it has seen the Space building or starting. If no build
# becomes visible within a few minutes (for example because nothing needed rebuilding), the state at that
# moment is taken as final. On a build or runtime error the last lines of the logs are printed.
#

def print_log_tail(api: HfApi, cfg: Config, lines: int = 30) -> None:
    """Print the last lines of the build log and the container log, to show why the Space failed."""
    for build in (True, False):
        try:
            log = list(api.fetch_space_logs(cfg.hf_space_repo, build=build))
        except Exception as error:  # a missing log must not hide the original failure
            print(f"  (could not read {'build' if build else 'container'} log: {error})")
            continue
        print(f"--- last {lines} lines of the {'build' if build else 'container'} log ---")
        print("\n".join(line.rstrip() for line in log[-lines:]))


def wait_until_running(api: HfApi, cfg: Config, expect_restart: bool) -> None:
    """Follow the Space status until the (new) app runs; raise on failure or timeout."""
    start = time.monotonic()
    deadline = start + cfg.space_build_timeout_s
    seen_activity = not expect_restart  # nothing changed -> the current state is already the final one
    last_stage = None
    while time.monotonic() < deadline:
        stage = api.get_space_runtime(cfg.hf_space_repo).stage
        if stage != last_stage:
            print(f"  Space status: {stage}", flush=True)
            last_stage = stage

        if stage in BUSY_STAGES:
            seen_activity = True
        # Trust the status once a build/restart was seen, or when none appeared within the grace time.
        trusted = seen_activity or time.monotonic() - start > RESTART_GRACE_S
        if trusted and stage == "RUNNING":
            return
        if trusted and stage in FAILED_STAGES:
            print_log_tail(api, cfg)
            raise RuntimeError(f"The Space stopped with status {stage}; see the log above.")
        time.sleep(cfg.space_poll_interval_s)

    print_log_tail(api, cfg)
    raise TimeoutError(f"The Space was not running after {cfg.space_build_timeout_s} seconds.")


def check_public_health(cfg: Config, attempts: int = 10) -> None:
    """Confirm the public address answers, without any login, the way a visitor would reach it."""
    url = f"{cfg.space_url}/_stcore/health"
    for attempt in range(1, attempts + 1):
        try:
            with urllib.request.urlopen(url, timeout=20) as response:
                if response.status == 200:
                    print(f"  Health check: {url} -> 200 OK")
                    return
        except (urllib.error.URLError, TimeoutError):
            pass  # the proxy may need a few more seconds after the status turns RUNNING
        time.sleep(cfg.space_poll_interval_s)
    raise RuntimeError(f"The app is running but {url} did not answer after {attempts} attempts.")


# =============================================================================
# Run all steps
# =============================================================================

def main() -> None:
    """Run steps 1 to 5 in order and print where the app can be reached."""
    cfg = Config()
    api = HfApi()  # write access, from HF_TOKEN or `hf auth login`

    # 1. Space
    ensure_public_space(api, cfg)

    # 2. Model version
    model_revision = resolve_model_revision(api, cfg)
    print(f"Space          : https://huggingface.co/spaces/{cfg.hf_space_repo}")
    print(f"Model version  : {cfg.hf_model_repo} @ {model_revision[:8]}")

    # 3. Space settings
    variables_changed = configure_space(api, cfg, model_revision)

    # 4. Upload
    space_commit, files_changed = upload_deployment_files(api, cfg)
    print(f"Space commit   : {space_commit[:8]}{'' if files_changed else '  (files unchanged, no new commit)'}")
    print(f"Uploaded files : {', '.join(cfg.deployment_files)}")

    # 5. Wait for the app
    print("Waiting for the Space to build and start ...", flush=True)
    wait_until_running(api, cfg, expect_restart=variables_changed or files_changed)
    check_public_health(cfg)
    print(f"\nApp is live    : {cfg.space_url}")


if __name__ == "__main__":
    main()

Overwriting tourism_project/hosting/hosting.py


In [53]:
# === PHASE 07b: DEPLOY TO HUGGING FACE SPACE ===
# Run the script written above: set the model version, upload the deployment files to the Space and wait
# until the app is live. %run stops the cell on an error, e.g. a failed build.
%run -m tourism_project.hosting.hosting

Space          : https://huggingface.co/spaces/richvrb/tourism-wellness-app
Model version  : richvrb/tourism-wellness-model @ d64d35ec


No files have been modified since last commit. Skipping to prevent empty commit.


Space commit   : 8637d18e  (files unchanged, no new commit)
Uploaded files : Dockerfile, .dockerignore, README.md, requirements.txt, app.py, .streamlit/config.toml
Waiting for the Space to build and start ...
  Space status: RUNNING_BUILDING
  Space status: RUNNING_APP_STARTING
  Space status: RUNNING
  Health check: https://richvrb-tourism-wellness-app.hf.space/_stcore/health -> 200 OK

App is live    : https://richvrb-tourism-wellness-app.hf.space


# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

Instead of copying the sample above by hand, the cells below write the workflow from this notebook, just like
every other file, so the notebook stays the single source of what ends up in the repository.

**How the pipeline works.** The workflow runs the same four scripts as this notebook, each as its own job on a
fresh GitHub machine:

| Job | Script | Passes on |
|---|---|---|
| `register-dataset` | `data_register.py`: raw data to the private dataset repository | |
| `data-prep` | `prep.py`: cleaning, grouping near-copies, group-aware train/test split | `data_revision` |
| `model-training` | `train.py`: compare candidates, select, test once, register the model | `model_revision` |
| `deploy-hosting` | `hosting.py`: push the app files to the Space, which builds and starts the app | |

- **Trigger.** Every push to `main` that changes `tourism_project/` or the workflow starts a run automatically; it
  can also be started by hand from the *Actions* tab. A change to only this notebook does not retrain the model.
- **Versions between jobs.** Jobs do not share a disk; the Hugging Face Hub is the shared storage. Each job passes
  the exact commit it produced to the next one, so training uses exactly the split that was just made, and the
  app loads exactly the model that was just registered, even if something else is uploaded in the meantime.
- **Secret.** The jobs use the GitHub secret `HF_TOKEN` (write access). It never appears in a file or a log.
- **Experiment results.** The candidates table, the selection and the test results appear on the summary page of
  each workflow run. The MLflow database is kept as the run artifact `mlflow`; download it and open it with
  `mlflow ui --backend-store-uri sqlite:///mlflow.db`. The template starts `mlflow ui` inside the job instead,
  but nobody can open a server on a GitHub machine, so that step is left out.

Compared with the sample YAML above, the workflow also pins Python 3.12 and uses current action versions, and
installs the packages through one shared setup step so all jobs use the same environment.

In [54]:
# === PHASE 08: WORKFLOW FOLDERS ===
# GitHub only runs workflows from .github/workflows in the repository root. The shared setup step used by
# every job lives in .github/actions/setup-pipeline. %%writefile does not create missing folders.
os.makedirs(".github/workflows", exist_ok=True)
os.makedirs(".github/actions/setup-pipeline", exist_ok=True)

print("Workflow folders ready: .github/workflows, .github/actions/setup-pipeline")

Workflow folders ready: .github/workflows, .github/actions/setup-pipeline


**Shared setup.** Every job starts on an empty machine and has to install Python and the packages again. This small
local action does that in one place, so a version change is made once and all four jobs stay identical.

In [55]:
%%writefile .github/actions/setup-pipeline/action.yml
# Shared setup for every job of the pipeline: the same Python version and the same pinned packages.
#
# Every job in GitHub Actions starts on a fresh machine, so each job has to install Python and the
# packages again. Keeping these steps in one place means a version change is made once and all jobs
# stay identical. A job uses it after checking out the repository:
#
#   - uses: actions/checkout@v7
#   - uses: ./.github/actions/setup-pipeline

name: Set up the pipeline environment
description: Install Python 3.12 and the pinned packages from tourism_project/requirements.txt

runs:
  using: composite
  steps:
    # Python 3.12 is required: numpy 2.5 and xgboost 3.4 do not support older versions. The pip cache
    # makes later runs faster; it is refreshed whenever requirements.txt changes.
    - name: Set up Python 3.12
      uses: actions/setup-python@v7
      with:
        python-version: "3.12"
        cache: pip
        cache-dependency-path: tourism_project/requirements.txt

    # Exact versions, so every run trains and scores with the same environment.
    - name: Install pinned packages
      shell: bash
      run: pip install -r tourism_project/requirements.txt

Overwriting .github/actions/setup-pipeline/action.yml


**The workflow.** Every job and step carries a comment explaining why it is there.

In [56]:
%%writefile .github/workflows/pipeline.yml
# MLOps pipeline for the Visit with Us Wellness Tourism model.
#
# Every change to the pipeline code on the main branch runs the full workflow automatically:
#
#   register-dataset  ->  data-prep  ->  model-training  ->  deploy-hosting
#   raw data on Hub       train/test     model on Hub        app in the public Space
#
# The jobs run the same scripts as the notebook (tourism.ipynb). The Hugging Face Hub is the shared storage
# between them: each job reads its input from the Hub and writes its output back, and the exact version it
# produced (a commit id) is passed on to the next job.
#
# Required secret: HF_TOKEN, a Hugging Face token with write access to the dataset repository, the model
# repository and the Space (Settings -> Secrets and variables -> Actions in the GitHub repository).

name: Tourism Project Pipeline

on:
  # Automatic run on every push to main that changes the pipeline code or this workflow. Changes that do
  # not affect the model, such as the notebook or documentation, do not start a new training run.
  push:
    branches:
      - main
    paths:
      - "tourism_project/**"
      - ".github/**"
  # Manual run from the Actions tab, e.g. to rebuild everything without a code change.
  workflow_dispatch:

# The jobs only need to read the repository; everything they write goes to the Hugging Face Hub.
permissions:
  contents: read

# One pipeline at a time: two runs uploading to the same Hub repositories at once could interleave.
# A newer push waits for the running pipeline instead of cancelling it halfway through an upload.
concurrency:
  group: tourism-pipeline
  cancel-in-progress: false

env:
  PYTHONUNBUFFERED: "1"                # show script output in the log as it happens
  HF_HUB_DISABLE_PROGRESS_BARS: "1"    # keep upload progress bars out of the log

jobs:

  # ---------------------------------------------------------------------------------------------------------
  # 1. Register the raw dataset (tourism_project/data/tourism.csv) in the private Hugging Face dataset repo.
  #    Unchanged data creates no new commit.
  # ---------------------------------------------------------------------------------------------------------
  register-dataset:
    runs-on: ubuntu-latest
    timeout-minutes: 15
    steps:
      - uses: actions/checkout@v7
      - uses: ./.github/actions/setup-pipeline

      - name: Upload dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python -m tourism_project.model_building.data_register

  # ---------------------------------------------------------------------------------------------------------
  # 2. Clean the data, group near-copies of records, make the group-aware train/test split and upload both
  #    files. The commit id of that upload (data_revision) is passed on to training.
  # ---------------------------------------------------------------------------------------------------------
  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    timeout-minutes: 15
    outputs:
      data_revision: ${{ steps.prep.outputs.data_revision }}
    steps:
      - uses: actions/checkout@v7
      - uses: ./.github/actions/setup-pipeline

      - name: Run data preparation
        id: prep   # prep.py writes data_revision to GITHUB_OUTPUT
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python -m tourism_project.model_building.prep

  # ---------------------------------------------------------------------------------------------------------
  # 3. Compare the candidate models, select one, evaluate it once on the test set and register it in the
  #    private model repository. The commit id of the registered model (model_revision) goes to hosting.
  #
  #    Experiment tracking: every run is logged to MLflow in outputs/mlflow.db. The candidates table, the
  #    selection and the test results appear on this workflow run's summary page; the MLflow database is
  #    kept as the artifact "mlflow" and can be opened locally with
  #    `mlflow ui --backend-store-uri sqlite:///mlflow.db`. No MLflow server is started here, because
  #    nobody could open it on a CI machine.
  # ---------------------------------------------------------------------------------------------------------
  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    timeout-minutes: 60
    outputs:
      model_revision: ${{ steps.train.outputs.model_revision }}
    steps:
      - uses: actions/checkout@v7
      - uses: ./.github/actions/setup-pipeline

      - name: Train, select and register the model
        id: train   # train.py writes model_revision to GITHUB_OUTPUT and tables to the run summary
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          DATA_REVISION: ${{ needs.data-prep.outputs.data_revision }}   # train on exactly this split
        run: python -m tourism_project.model_building.train

      # Kept even when training fails, so a failed run can be investigated. Contains aggregated metrics and
      # parameters only, no customer rows.
      - name: Keep the MLflow experiment database
        if: always()
        uses: actions/upload-artifact@v7
        with:
          name: mlflow
          path: |
            outputs/mlflow.db
            outputs/model/operating_points.csv
          if-no-files-found: warn

  # ---------------------------------------------------------------------------------------------------------
  # 4. Push the deployment files to the public Hugging Face Space, with the Space variable MODEL_REVISION
  #    set to the model registered in job 3. Hugging Face builds the Docker image from these files;
  #    hosting.py waits until the app runs and checks that its public address answers.
  # ---------------------------------------------------------------------------------------------------------
  deploy-hosting:
    needs: [register-dataset, data-prep, model-training]
    runs-on: ubuntu-latest
    timeout-minutes: 30
    steps:
      - uses: actions/checkout@v7
      - uses: ./.github/actions/setup-pipeline

      - name: Push files to the Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          MODEL_REVISION: ${{ needs.model-training.outputs.model_revision }}   # serve exactly this model
        run: python -m tourism_project.hosting.hosting

Overwriting .github/workflows/pipeline.yml


## Requirements file for the Github Actions Workflow

**Why a separate requirements file.** The pipeline jobs need more than the app: SciPy for grouping near-copies and
MLflow for experiment tracking. Every version is pinned exactly, so each run trains in the same environment. The
first five packages must stay identical to `tourism_project/deployment/requirements.txt`: the model trained in the
pipeline has to load and score identically in the app, which the app checks when it starts.

In [57]:
%%writefile tourism_project/requirements.txt
# Packages for the pipeline jobs in GitHub Actions (data registration, preparation, training, hosting).
#
# Every version is pinned exactly, so each pipeline run uses the same environment and produces the same
# results. The model that training registers records these versions in model_metadata.json.
#
# The first five MUST stay identical to tourism_project/deployment/requirements.txt: the app has to load
# and score the model with the versions it was trained with. The app checks this when it starts.
pandas==3.0.3
numpy==2.5.1
scikit-learn==1.9.1
xgboost==3.4.1
joblib==1.6.0

# Used by data preparation (grouping near-copies of records).
scipy==1.18.0

# Experiment tracking during training, and access to the Hugging Face Hub in every job.
mlflow==3.16.0
huggingface_hub==1.31.0

Overwriting tourism_project/requirements.txt


## Github Authentication and Push Files

**How the files reach GitHub.** The files this notebook writes (`tourism_project/...` and `.github/...`) are pushed to the
GitHub repository `richardverbrugge-oss/tourism` from a local checkout of that repository, for example
with *Source Control* in VS Code, or `git add`, `git commit` and `git push` in a terminal.

The template pushes from inside the notebook instead, with a personal access token written into the
`git push` command. That is avoided here on purpose: this notebook is published in a public repository,
so a token in a cell or its output would be exposed, and the template's `git config --global` commands
would overwrite the Git identity of whoever runs the cell locally. A push from the local checkout uses
the credentials already stored there and needs no token in any file.

The push to `main` starts the GitHub Actions workflow, which runs the same scripts as this notebook.

In [58]:
# === PHASE 09: FILES TO PUSH TO GITHUB ===
# List the project and workflow files that belong in the GitHub repository. This cell only reads; it changes
# nothing. os.walk skips folders it cannot read, such as sandbox placeholders some Windows tools create.
project_files = sorted(
    os.path.join(folder, name)
    for root in ("tourism_project", ".github")
    for folder, subfolders, names in os.walk(root)
    for name in names
    if "__pycache__" not in folder and not name.startswith("~")
)
print(f"{len(project_files)} files in tourism_project/ and .github/ to commit and push:")
for path in project_files:
    print(f"  {path}")
print("\nPush them from the local checkout of the repository (VS Code Source Control or git push).")
print("The push to main starts the GitHub Actions pipeline.")

20 files in tourism_project/ and .github/ to commit and push:
  .github/actions/setup-pipeline/action.yml
  .github/workflows/pipeline.yml
  tourism_project/__init__.py
  tourism_project/ci.py
  tourism_project/config.py
  tourism_project/data/tourism.csv
  tourism_project/deployment/.dockerignore
  tourism_project/deployment/.streamlit/config.toml
  tourism_project/deployment/Dockerfile
  tourism_project/deployment/README.md
  tourism_project/deployment/app.py
  tourism_project/deployment/requirements.txt
  tourism_project/hosting/__init__.py
  tourism_project/hosting/hosting.py
  tourism_project/hub.py
  tourism_project/model_building/__init__.py
  tourism_project/model_building/data_register.py
  tourism_project/model_building/prep.py
  tourism_project/model_building/train.py
  tourism_project/requirements.txt

Push them from the local checkout of the repository (VS Code Source Control or git push).
The push to main starts the GitHub Actions pipeline.


# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

<font size=6 color="navyblue">Power Ahead!</font>
___